# Midterm Project: Quant Modeling for Data Scientists

In [2]:
import re
from typing import Tuple, Optional, List, Dict, Any, Union
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from geopy.distance import geodesic
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np
from typing import Optional

from geopy.distance import geodesic
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np
from typing import Optional, Tuple
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
pd.set_option('display.max_rows', None)

In [3]:
# 读取并查看房价数据
df_price = pd.read_csv("data/ruc_Class25Q2_train_price.csv")
print("房价数据概览:")
print("数据维度:", df_price.shape)
print("\n前3行数据:")
print(df_price.head(3))

房价数据概览:
数据维度: (103871, 55)

前3行数据:
   城市     区域     板块    环线         Price      房屋户型       所在楼层     建筑面积  \
0   0  109.0  150.0  二至三环  6.194049e+06  2室1厅1厨1卫  中楼层 (共5层)    52.3㎡   
1   0   65.0  299.0  五至六环  4.354153e+06  3室1厅1厨1卫   顶层 (共6层)  127.44㎡   
2   0   62.0  911.0  五至六环  3.321992e+06  3室2厅1厨2卫  低楼层 (共6层)  118.02㎡   

      套内面积 房屋朝向  ...     供水        供暖     供电       燃气费    供热费     停车位 停车费用  \
0      NaN  南 北  ...     民水      集中供暖     民电  2.61元/m³  30元/㎡   300.0   暂无   
1   123.7㎡  南 北  ...  商水/民水       自采暖  商电/民电  2.61元/m³    NaN  1550.0  150   
2  101.95㎡   东南  ...  商水/民水  集中供暖/自采暖  商电/民电  2.61元/m³  30元/㎡   324.0  150   

      coord_x    coord_y                   客户反馈  
0  117.424278  40.975752           听说，设施老旧，停车费高  
1  117.389228  41.091295          整体印象，网速快，面积适中  
2  117.200934  40.747919  地段一般，停车划线清晰，说白了，居住体验佳  

[3 rows x 55 columns]


C:\Users\lenovo\AppData\Local\Temp\ipykernel_19996\3865851362.py:2: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df_price = pd.read_csv("data/ruc_Class25Q2_train_price.csv")


In [4]:
print("\n房价数据信息:")
df_price.info()

print("\n房价数据缺失值统计:")
print(df_price.isnull().sum().sort_values(ascending=False))


房价数据信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103871 entries, 0 to 103870
Data columns (total 55 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   城市         103871 non-null  int64  
 1   区域         103871 non-null  float64
 2   板块         103871 non-null  float64
 3   环线         40419 non-null   object 
 4   Price      103871 non-null  float64
 5   房屋户型       103291 non-null  object 
 6   所在楼层       103871 non-null  object 
 7   建筑面积       103871 non-null  object 
 8   套内面积       35984 non-null   object 
 9   房屋朝向       103870 non-null  object 
 10  建筑结构       103291 non-null  object 
 11  装修情况       103291 non-null  object 
 12  梯户比例       101252 non-null  object 
 13  配备电梯       91520 non-null   object 
 14  别墅类型       1443 non-null    object 
 15  交易时间       103871 non-null  object 
 16  交易权属       103871 non-null  object 
 17  上次交易       78422 non-null   object 
 18  房屋用途       103870 non-null  object 
 19  房屋年限       593

In [5]:
# 读取并查看租金数据
df_rent = pd.read_csv("data/ruc_Class25Q2_train_rent.csv")
print("租金数据概览:")
print("数据维度:", df_rent.shape)
print("\n前3行数据:")
print(df_rent.head(3))

租金数据概览:
数据维度: (98899, 46)

前3行数据:
   城市      户型   装修          Price     楼层      面积 朝向        交易时间 付款方式 租赁方式  \
0   0  1室1厅1卫  精装修  654646.481811   4/6层  36.42㎡  西  2024-11-28  季付价   整租   
1   0  1室1厅1卫  精装修  665412.057415   4/6层  41.00㎡  南  2024-10-30  季付价   整租   
2   0  1室1厅1卫  精装修  778222.820548  1/18层  37.36㎡  北  2024-11-12  季付价   整租   

   ...     供水    供暖     供电            燃气费       供热费    停车位 停车费用     coord_x  \
0  ...     民水  集中供暖     民电       2.61元/m³  24-30元/㎡  450.0  150  117.339283   
1  ...     民水  集中供暖     民电       2.61元/m³     30元/㎡  150.0  150  117.446526   
2  ...  商水/民水  集中供暖  商电/民电  2.61-2.63元/m³  30-46元/㎡  965.0  500  117.518524   

     coord_y                    客户反馈  
0  40.930007          潮气重，仔细一看，房屋保养好  
1  40.876743  服务响应中等，看起来，管线老化，消防设施齐全  
2  40.905357  差不多这样，电梯新，总的来说，宽敞，性价比高  

[3 rows x 46 columns]


C:\Users\lenovo\AppData\Local\Temp\ipykernel_19996\237608512.py:2: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rent = pd.read_csv("data/ruc_Class25Q2_train_rent.csv")


In [6]:
print("\n租金数据信息:")
df_rent.info()

print("\n租金数据缺失值统计:")
print(df_rent.isnull().sum().sort_values(ascending=False))


租金数据信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98899 entries, 0 to 98898
Data columns (total 46 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   城市       98899 non-null  int64  
 1   户型       98898 non-null  object 
 2   装修       25410 non-null  object 
 3   Price    98899 non-null  float64
 4   楼层       98894 non-null  object 
 5   面积       98899 non-null  object 
 6   朝向       98894 non-null  object 
 7   交易时间     98899 non-null  object 
 8   付款方式     80476 non-null  object 
 9   租赁方式     98899 non-null  object 
 10  电梯       98895 non-null  object 
 11  车位       24764 non-null  object 
 12  用水       81159 non-null  object 
 13  用电       81575 non-null  object 
 14  燃气       94317 non-null  object 
 15  采暖       34412 non-null  object 
 16  租期       51966 non-null  object 
 17  配套设施     68448 non-null  object 
 18  lon      98899 non-null  float64
 19  lat      98899 non-null  float64
 20  年份       98899 non-null  float64
 21  区县 

In [7]:
#使用IQR方法移除DataFrame中价格列的异常值
def remove_price_outliers(df, price_column='Price', iqr_multiplier=1.5, inplace=False):
    """
    使用IQR方法移除DataFrame中价格列的异常值
    
    参数:
    df: 输入的DataFrame
    price_column: 价格列的名称，默认为'Price'
    iqr_multiplier: IQR倍数，默认为1.5
    inplace: 是否在原DataFrame上修改，默认为False
    
    返回:
    清理后的DataFrame（如果inplace=False）
    或None（如果inplace=True）
    """
    print("移除价格异常值")
    
    # 计算IQR和边界
    Q1 = df[price_column].quantile(0.25)
    Q3 = df[price_column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - iqr_multiplier * IQR
    upper_bound = Q3 + iqr_multiplier * IQR
    
    # 移除异常值
    df_cleaned = df[(df[price_column] >= lower_bound) & 
                    (df[price_column] <= upper_bound)]
    
    print(f"原始数据行数: {len(df)}")
    print(f"移除异常值之后的行数: {len(df_cleaned)}")
    print(f"移除了 {len(df) - len(df_cleaned)} 行异常数据")
    print(f"价格范围: [{lower_bound:.2f}, {upper_bound:.2f}]")
    
    if inplace:
        df = df_cleaned
        return None
    else:
        return df_cleaned

In [8]:
df_price=remove_price_outliers(df_price)
print("移除价格异常值")
print("df_price数据维度:", df_price.shape)

df_rent=remove_price_outliers(df_rent)
print("移除租金异常值")
print("df_rent数据维度:", df_rent.shape)


移除价格异常值
原始数据行数: 103871
移除异常值之后的行数: 96048
移除了 7823 行异常数据
价格范围: [-1793407.31, 5365255.64]
移除价格异常值
df_price数据维度: (96048, 55)
移除价格异常值
原始数据行数: 98899
移除异常值之后的行数: 93365
移除了 5534 行异常数据
价格范围: [-483272.71, 1449796.34]
移除租金异常值
df_rent数据维度: (93365, 46)


In [9]:

# ==================== 辅助函数 ====================

def hanzi_to_num(hanzi: str) -> Optional[int]:
    """汉字数字转换"""
    if not isinstance(hanzi, str) or hanzi.strip() == '':
        return None
    num_map = {'零':0,'一':1,'二':2,'两':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9}
    hanzi = hanzi.strip()
    if hanzi in num_map:
        return num_map[hanzi]
    if '十' in hanzi:
        parts = hanzi.split('十')
        left = parts[0]
        right = parts[1] if len(parts) > 1 else ''
        left_val = num_map.get(left, 1) if left != '' else 1
        right_val = num_map.get(right, 0) if right != '' else 0
        return left_val * 10 + right_val
    total = 0
    for ch in hanzi:
        if ch in num_map:
            total = total*10 + num_map[ch]
        else:
            return None
    return total if total != 0 else None

def safe_log_transform(series: pd.Series, offset: float = 1.0) -> pd.Series:
    """安全的对数变换，处理0和负值"""
    return np.log(series + offset)

def create_interaction_terms(df: pd.DataFrame, col1: str, col2: str) -> pd.Series:
    """创建两个特征的交互项"""
    if col1 in df.columns and col2 in df.columns:
        return df[col1] * df[col2]
    return pd.Series(index=df.index)



## price

In [10]:
# 初始化空的X
X = pd.DataFrame()

In [11]:
def price_process_city(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理城市列 - 独热编码为布尔类型"""
    if '城市' in df_price.columns:
        city_col = df_price[['城市']].astype(str)
        city_dummies = pd.get_dummies(city_col['城市'], prefix='城市').astype(bool)
        X = pd.concat([X, city_dummies], axis=1)
        if verbose: 
            print("加入城市独热编码，新增列数：", city_dummies.shape[1])
    return X

In [12]:
def price_process_ring(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理环线列 - 映射分数"""
    if '环线' in df_price.columns:
        ring_score = {
            '内环内': 10,       # 最核心区域，分数最高
            '内环至中环': 9,    # 内环向外延伸至中环，次核心
            '二环内': 8,        # 二环以内（通常二环略外于内环，分数稍低）
            '内环至外环': 7,    # 覆盖内环到外环的中间区域，比中环至外环更靠近核心
            '中环至外环': 6,    # 中环向外延伸至外环
            '二至三环': 5,      # 二环外、三环内
            '三至四环': 4,      # 三环外、四环内
            '四至五环': 3,      # 四环外、五环内
            '五至六环': 2,      # 五环外、六环内
            '六环外': 1,        # 六环以外
            '外环外': 0         # 外环以外（最外围，分数最低）
        }
        df_price['环线_score'] = df_price['环线'].map(ring_score)
        df_price['环线_score'] = pd.to_numeric(df_price['环线_score'], errors='coerce')
        median_ring = df_price['环线_score'].median()
        df_price['环线_score'] = df_price['环线_score'].fillna(median_ring)
        X['环线'] = df_price['环线_score']
        if verbose: print("加入环线（中位数填充）")
    return X

In [13]:
def price_process_room_layout(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """处理房屋户型列 - 解析室/厅/厨/卫 + 创建比率特征"""
    def parse_room_info(x):
        room, hall, kitchen, bath = pd.NA, pd.NA, pd.NA, pd.NA
        if not pd.isna(x):
            s = str(x)
            room_match = re.search(r'(\d+)(室|房间)', s)
            hall_match = re.search(r'(\d+)厅', s)
            kitchen_match = re.search(r'(\d+)厨', s)
            bath_match = re.search(r'(\d+)卫', s)
            room = int(room_match.group(1)) if room_match else 0
            hall = int(hall_match.group(1)) if hall_match else 0
            kitchen = int(kitchen_match.group(1)) if kitchen_match else 0
            bath = int(bath_match.group(1)) if bath_match else 0
        return pd.Series([room, hall, kitchen, bath], index=['室', '厅', '厨', '卫'])

    if '房屋户型' in df_price.columns:
        room_features = df_price['房屋户型'].apply(parse_room_info)
        room_features = room_features.astype({'室': 'Int64', '厅': 'Int64', '厨': 'Int64', '卫': 'Int64'})
        
        # 填充缺失值
        for col in ['室','厅','厨','卫']:
            median_val = room_features[col].median()
            room_features[col] = room_features[col].fillna(median_val).astype('Int64')
        
        # 基础特征
        X = pd.concat([X, room_features], axis=1)
        
        # 创建房间总数
        X['房间总数'] = room_features['室'] + room_features['厅']
        
        # 创建卫生间/卧室比率（重要特征）
        X['卫室比'] = room_features['卫'] / (room_features['室'] + 0.01)  # 避免除0
        
        # 创建是否有厨房的布尔特征
        X['有厨房'] = (room_features['厨'] > 0).astype(bool)
        
        df_price = df_price.drop(columns=['房屋户型'])
        if verbose: 
            print("加入房屋户型拆分特征：室/厅/厨/卫 + 比率特征")
    return X, df_price

In [14]:
def price_process_floor(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理所在楼层列 - 解析楼层类型和总楼层"""
    def parse_floor(x):
        if pd.isna(x):
            return pd.Series([None, None], index=['楼层类型','总楼层'])
        s = str(x)
        match = re.search(r'共(\d+)层', s)
        total = int(match.group(1)) if match else None
        if '底' in s:
            level_type = '底'
        elif '低' in s:
            level_type = '低'
        elif '中' in s:
            level_type = '中'
        elif '高' in s:
            level_type = '高'
        elif '顶' in s:
            level_type = '顶'
        elif '地下' in s:
            level_type = '地下'
        else:
            level_type = '未知'
        return pd.Series([level_type, total], index=['楼层类型','总楼层'])

    if '所在楼层' in df_price.columns:
        parsed_features = df_price['所在楼层'].apply(parse_floor)
        if parsed_features['总楼层'].isna().any():
            parsed_features['总楼层'] = parsed_features['总楼层'].fillna(parsed_features['总楼层'].median())
        parsed_features['楼层类型'] = parsed_features['楼层类型'].fillna('未知')
        floor_dummies = pd.get_dummies(parsed_features['楼层类型'], prefix='楼层', drop_first=False).astype(bool)
        X = pd.concat([X, parsed_features[['总楼层']], floor_dummies], axis=1)
        if verbose: print("加入所在楼层特征：总楼层 + 楼层类型独热编码")
    return X



def price_process_area(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理建筑面积列 - 数值化 + 对数变换 + 分箱"""
    if '建筑面积' in df_price.columns:
        area_series = df_price['建筑面积'].astype(str).str.replace('㎡','', regex=False).str.strip()
        area_numeric = pd.to_numeric(area_series.replace({'nan': np.nan}), errors='coerce')
        
        # 基础面积特征
        X['建筑面积'] = area_numeric
        
        # 面积对数变换（处理偏态分布）
        X['建筑面积_log'] = safe_log_transform(area_numeric.fillna(area_numeric.median()))
        
        # 面积平方项
        X['建筑面积_sq'] = area_numeric ** 2
        
        if verbose: 
            print("加入建筑面积特征：数值 + 对数 + 平方")
    return X

In [15]:
def price_process_direction(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理房屋朝向列 - 八方向布尔特征 + 创建朝向得分"""
    if '房屋朝向' in df_price.columns:
        directions = ['东', '南', '西', '北', '东南', '西南', '东北', '西北']
        direction_features = pd.DataFrame(index=df_price.index)
        
        # 基础方向特征
        for d in directions:
            direction_features[f'朝向_{d}'] = df_price['房屋朝向'].apply(lambda x: 1 if d in str(x) else 0)
        direction_features = direction_features.astype(bool)
        
        # 创建朝向得分（南=3, 东/东南=2, 北/西=1, 其他=0）
        direction_score = pd.Series(0, index=df_price.index)
        direction_score += direction_features['朝向_南'] * 3
        direction_score += direction_features['朝向_东南'] * 2
        direction_score += direction_features['朝向_东'] * 2
        direction_score += direction_features['朝向_西南'] * 1
        direction_score += direction_features['朝向_东北'] * 1
        direction_score += direction_features['朝向_北'] * 1
        direction_score += direction_features['朝向_西'] * 1
        
        X['朝向得分'] = direction_score
        X = pd.concat([X, direction_features], axis=1)
        
        if verbose: 
            print("加入朝向特征：八方向 + 综合得分")
    return X


In [16]:
def price_process_structure(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理建筑结构列 - 独热编码为布尔类型"""
    if '建筑结构' in df_price.columns:
        structure_series = df_price['建筑结构'].fillna('未知结构').astype(str)
        structure_dummies = pd.get_dummies(structure_series, prefix='结构', drop_first=False).astype(bool)
        X = pd.concat([X, structure_dummies], axis=1)
        if verbose: 
            print("加入建筑结构独热编码")
    return X

def price_process_decoration(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理装修情况列 - 独热编码为布尔类型 + 创建装修等级"""
    if '装修情况' in df_price.columns:
        decoration_series = df_price['装修情况'].fillna('毛坯').astype(str)
        
        # 基础独热编码
        decoration_dummies = pd.get_dummies(decoration_series, prefix='装修', drop_first=False).astype(bool)
        
        # 创建装修等级（数值特征）
        decoration_grade = {
            '毛坯': 1,
            '简装': 2, 
            '精装': 4,
        }
        # 映射装修等级，未知的默认为2（简装）
        X['装修等级'] = decoration_series.map(decoration_grade).fillna(2)
        
        X = pd.concat([X, decoration_dummies], axis=1)
        if verbose: 
            print("加入装修情况特征：独热编码 + 等级评分")
    return X

In [17]:
def price_process_elevator_ratio(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理梯户比例列 - 解析梯数和户数 + 创建梯户比"""
    def parse_ratio(x):
        if pd.isna(x): 
            return pd.Series([np.nan, np.nan], index=['梯数','户数'])
        s = str(x)
        match = re.search(r'(.+?)梯(.+?)户', s)
        if match:
            lift_hanzi = match.group(1)
            household_hanzi = match.group(2)
            lift_num = hanzi_to_num(lift_hanzi)
            household_num = hanzi_to_num(household_hanzi)
            return pd.Series([lift_num, household_num], index=['梯数','户数'])
        else:
            return pd.Series([np.nan, np.nan], index=['梯数','户数'])

    if '梯户比例' in df_price.columns:
        parsed_ratio = df_price['梯户比例'].apply(parse_ratio)
        for col in ['梯数','户数']:
            if parsed_ratio[col].notna().any():
                parsed_ratio[col] = parsed_ratio[col].fillna(parsed_ratio[col].median())
            else:
                parsed_ratio[col] = 0
        
        # 基础特征
        X = pd.concat([X, parsed_ratio[['梯数','户数']]], axis=1)
        
        # 创建梯户比（重要特征）
        X['梯户比'] = parsed_ratio['户数'] / (parsed_ratio['梯数'] + 0.01)
        
        
        if verbose: 
            print("加入梯户特征：梯数、户数 + 梯户比")
    return X


def price_process_villa_type(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理别墅类型列 - 数值映射 + 是否为别墅标志"""
    if '别墅类型' in df_price.columns:
        villa_type_map = {'独栋':4,'双拼':3,'联排':2,'叠拼':1}
        X['别墅类型'] = df_price['别墅类型'].map(villa_type_map).fillna(0).astype(int)
        if verbose: 
            print("加入别墅类型特征：数值编码 + 布尔标志")
    return X

In [18]:
def price_process_building_age(df_price: pd.DataFrame, X: pd.DataFrame, current_year: int = 2025, verbose: bool = True) -> pd.DataFrame:
    """处理建筑年代列 - 计算建筑年龄 + 分箱 + 交互特征"""
    if '建筑年代' in df_price.columns:
        def parse_year(x):
            if pd.isna(x): return np.nan
            s = str(x)
            nums = re.findall(r'\d{4}', s)
            if len(nums) == 1:
                return int(nums[0])
            elif len(nums) == 2:
                return int((int(nums[0]) + int(nums[1])) / 2)
            else:
                return np.nan
        df_price['建筑年代_parsed'] = df_price['建筑年代'].apply(parse_year)
        median_year = int(df_price['建筑年代_parsed'].median())
        df_price['建筑年代_parsed'] = df_price['建筑年代_parsed'].fillna(median_year)
        
        # 基础建筑年龄
        building_age = current_year - df_price['建筑年代_parsed']
        X['建筑年龄'] = building_age

        # 是否为新房（5年内）
        X['是新房'] = (building_age <= 5).astype(bool)
        
        if verbose: 
            print("加入建筑年龄特征：数值 + 分箱 + 新房标志")
    return X


def price_process_building_counts(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理房屋总数和楼栋总数 - 数值化 + 创建密度特征"""
    # 房屋总数
    if '房屋总数' in df_price.columns:
        X['房屋总数'] = df_price['房屋总数'].str.replace('户', '', regex=False).astype(float)
        X['房屋总数'] = X['房屋总数'].fillna(X['房屋总数'].median())
    
    # 楼栋总数
    if '楼栋总数' in df_price.columns:
        X['楼栋总数'] = df_price['楼栋总数'].str.replace('栋', '', regex=False).astype(float)
        X['楼栋总数'] = X['楼栋总数'].fillna(X['楼栋总数'].median())
    
    # 特征工程优化 - 创建密度特征
    if '房屋总数' in X.columns and '楼栋总数' in X.columns:
        X['每栋户数'] = X['房屋总数'] / (X['楼栋总数'] + 0.01)
        X['每栋户数_log'] = safe_log_transform(X['每栋户数'])
    
    if verbose and ('房屋总数' in df_price.columns or '楼栋总数' in df_price.columns):
        print("加入楼栋房屋特征：数值 + 密度特征")
    
    return X

In [19]:
def price_process_green_rate(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理绿化率列 - 数值化 + 分箱"""
    if '绿 化 率' in df_price.columns:
        green = df_price['绿 化 率'].astype(str).str.replace('%','',regex=False).replace({'nan': np.nan})
        df_price['绿化率_num'] = pd.to_numeric(green, errors='coerce')
        normal_green = df_price[(df_price['绿化率_num'].notna()) & (df_price['绿化率_num'] <= 100)]['绿化率_num']
        median_green = normal_green.median() if not normal_green.empty else 0
        df_price['绿化率_num'] = df_price['绿化率_num'].apply(lambda x: median_green if (pd.isna(x) or (x>100)) else x)
        
        # 基础绿化率
        X['绿化率'] = df_price['绿化率_num']
        
        if verbose: 
            print("加入绿化率特征：数值")
    return X


def price_process_floor_area_ratio(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理容积率列 - 中位数填充 + 分箱"""
    if '容 积 率' in df_price.columns:
        median_far = df_price['容 积 率'].median()
        far_filled = df_price['容 积 率'].fillna(median_far)
        
        # 基础容积率
        X['容积率'] = far_filled
        
        if verbose: 
            print("加入容积率特征：数值 + 分箱")
    return X

In [20]:
def price_process_property_fee(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理物业费列 - 解析数值 + 对数变换 + 分箱"""
    if '物 业 费' in df_price.columns:
        def parse_fee(x):
            if pd.isna(x): return np.nan
            s = str(x).replace('元/月/㎡','').strip()
            if '-' in s:
                parts = s.split('-')
                try:
                    num1 = float(parts[0])
                    num2 = float(parts[1])
                    return (num1 + num2) / 2
                except:
                    return np.nan
            else:
                try:
                    return float(s)
                except:
                    return np.nan
        df_price['物业费_num'] = df_price['物 业 费'].apply(parse_fee)
        median_fee = df_price['物业费_num'].median() if df_price['物业费_num'].notna().any() else 0
        fee_filled = df_price['物业费_num'].fillna(median_fee)
        
        # 基础物业费
        X['物业费'] = fee_filled
        
        # 物业费对数变换
        X['物业费_log'] = safe_log_transform(fee_filled)
        
        if verbose: 
            print("加入物业费特征：数值 + 对数")
    return X

In [21]:
def create_interaction_features(X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """创建特征交互项"""
    
    # 面积与房间数的交互
    if '建筑面积' in X.columns and '室' in X.columns:
        X['每室面积'] = X['建筑面积'] / (X['室'] + 0.01)
        X['每厅面积'] = X['建筑面积'] / (X['厅'] + 0.01)
    
    # 建筑年龄与装修等级的交互
    if '建筑年龄' in X.columns and '装修等级' in X.columns:
        X['老房新装'] = ((X['建筑年龄'] > 20) & (X['装修等级'] >= 4)).astype(bool)
    
    # 别墅类型与面积的交互
    if '是别墅' in X.columns and '建筑面积' in X.columns:
        X['别墅面积交互'] = X['是别墅'] * X['建筑面积']
    
    # 楼层位置与面积的交互
    if '楼层位置比' in X.columns and '建筑面积' in X.columns:
        X['楼层面积交互'] = X['楼层位置比'] * X['建筑面积']
    
    if verbose:
        print("创建交互特征完成")
    
    return X

In [22]:
def price_process_house_age_limit(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理房屋年限列 - 数值编码"""
    if '房屋年限' in df_price.columns:
        year_limit_map = {'满五年':3,'满两年':2,'未满两年':1}
        X['房屋年限'] = df_price['房屋年限'].map(year_limit_map).fillna(3).astype(int)
        if verbose: print("加入房屋年限编码")
    return X

def price_process_transaction_type(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理交易权属列 - 布尔特征"""
    if '交易权属' in df_price.columns:
        X['交易权属'] = np.where(df_price['交易权属'] == '商品房', 1, 0).astype(bool)
        if verbose: print("加入交易权属布尔特征")
    return X

def price_process_property_ownership(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理产权所属列 - 布尔映射"""
    if '产权所属' in df_price.columns:
        property_ownership_map = {'共有':0, '非共有':1}
        X['产权所属'] = df_price['产权所属'].map(property_ownership_map).fillna(0).astype(bool)
        if verbose: print("加入产权所属布尔映射")
    return X


def price_process_water_supply(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理供水列 - 民/商布尔标志"""
    if '供水' in df_price.columns:
        X['供水_民'] = df_price['供水'].apply(lambda x: int('民' in str(x))).astype(bool)
        X['供水_商'] = df_price['供水'].apply(lambda x: int('商' in str(x))).astype(bool)
        if verbose: print("加入供水_民/供水_商")
    return X

def price_process_heating(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理供暖列 - 编码类型 + 独热编码"""
    if '供暖' in df_price.columns:
        # 填充空值
        heating_series = df_price['供暖'].fillna(0)
        
        # 编码函数
        def heating_to_num(text):
            if not isinstance(text, str):
                return 0
            if '集中供暖' in text:
                return 2
            elif '自采暖' in text:
                return 1
            else:
                return 0
        
        X['供暖_encoded'] = heating_series.apply(heating_to_num)
        
        # 供暖类型独热编码
        heating_dummies = pd.get_dummies(heating_series.astype(str), prefix='供暖').astype(bool)
        X = pd.concat([X, heating_dummies], axis=1)
        
        if verbose:
            print("加入供暖特征：有序编码 + 独热编码")
    
    return X

def price_process_power_supply(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理供电列 - 民/商布尔标志"""
    if '供电' in df_price.columns:
        X['供电_民'] = df_price['供电'].apply(lambda x: int('民' in str(x))).astype(bool)
        X['供电_商'] = df_price['供电'].apply(lambda x: int('商' in str(x))).astype(bool)
        if verbose: print("加入供电_民/供电_商")
    return X


def price_process_gas_fee(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理燃气费列 - 解析数值"""
    if '燃气费' in df_price.columns:
        def parse_gas_fee(x):
            if pd.isna(x): return np.nan
            s = str(x).replace('元/m³','').replace('元/m3', '').strip()
            if '-' in s:
                parts = s.split('-')
                try:
                    return (float(parts[0]) + float(parts[1]))/2
                except:
                    return np.nan
            else:
                try:
                    return float(s)
                except:
                    return np.nan
        df_price['燃气费_num'] = df_price['燃气费'].apply(parse_gas_fee)
        median_gas = df_price['燃气费_num'].median()
        df_price['燃气费_num'] = df_price['燃气费_num'].fillna(median_gas)
        X['燃气费'] = df_price['燃气费_num']
        if verbose: print("加入燃气费（数值化）")
    return X

def price_process_parking(df_price: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理停车位列 - 数值化"""
    if '停车位' in df_price.columns:
        df_price['停车位_num'] = pd.to_numeric(df_price['停车位'], errors='coerce')
        median_parking = df_price['停车位_num'].median() if df_price['停车位_num'].notna().any() else 0
        X['停车位'] = df_price['停车位_num'].fillna(median_parking)
        if verbose: print("加入停车位（数值化、缺失用中位数填充）")
    return X


In [23]:
def handle_district(df):
    """
    处理 '区县' 列 数字型分类特征。
    1. 填充 7% 的缺失值 (NaN) 为 '未知' 类别。
    2. 将整列转换为 'string' 类型，以防止模型将其误认为连续数值。
    """
    col = '区县'
    print(f"处理 [{col}]...")
    if col not in df.columns:
        print(f"警告: 列 '{col}' 不存在。")
        return df

    df[col] = df[col].fillna('未知')
    df[col] = df[col].astype(str)
    
    print(f"  已将 '{col}' 填充缺失值并转换为 'string' 类型。")
    print(f"  处理后 '{col}' 的唯一值 (示例): {df[col].unique()[:10]}")
    return df

def handle_plate(df):
    """
    处理 '板块' 列
    """
    col = '板块'
    print(f"处理 [{col}]...")
    if col not in df.columns:
        print(f"警告: 列 '{col}' 不存在。")
        return df

    df[col] = df[col].fillna('未知')
    df[col] = df[col].astype(str)
    
    print(f"  已将 '{col}' 填充缺失值并转换为 'string' 类型。")
    return df

In [24]:
def compute_city_center_and_distances(df, n_train, verbose=True):
    """计算城市中心点及房源到中心点的距离。"""
    df_out = df.copy()
    if verbose:
        print("开始计算地理空间特征 (距离中心)...")
    
    if '城市' not in df_out.columns or 'lon' not in df_out.columns or 'lat' not in df_out.columns:
        if verbose:
            print("  警告: 缺少 '城市', 'lon', 或 'lat' 列，无法计算距离特征。")
        return df_out

    train_part = df_out.iloc[:n_train] 
    city_centers = train_part.groupby('城市', observed=True)[['lon', 'lat']].mean().reset_index()
    city_centers = city_centers.rename(columns={'lon': 'center_lon', 'lat': 'center_lat'})
    if verbose:
        print(f"  计算了 {len(city_centers)} 个城市的中心点 (基于训练集)。")

    df_out = pd.merge(df_out, city_centers, on='城市', how='left')

    def compute_distance(row):
        if pd.isna(row['lat']) or pd.isna(row['lon']) or pd.isna(row['center_lat']) or pd.isna(row['center_lon']):
            return np.nan
        try:
            return geodesic((row['lat'], row['lon']), (row['center_lat'], row['center_lon'])).km
        except ValueError:
            return np.nan

    df_out['距离中心_公里'] = df_out.apply(compute_distance, axis=1)
    median_dist_train = df_out.iloc[:n_train]['距离中心_公里'].median()
    df_out['距离中心_公里'].fillna(median_dist_train, inplace=True)
    if verbose:
        print(f"  计算了 '距离中心_公里'，并用训练集中位数 ({median_dist_train:.2f} km) 填充了 NaN。")

    df_out['距离中心_公里_平方'] = df_out['距离中心_公里'] ** 2
    df_out = df_out.drop(columns=['center_lon', 'center_lat'], errors='ignore')
    if verbose:
        print("地理距离特征创建完毕。")
    return df_out

def create_geo_clusters(df, n_train, n_clusters=5, city_col='城市', lon_col='lon', lat_col='lat', verbose=True):
    """为每个城市计算地理聚类并进行独热编码。"""
    df_processed = df.copy()
    if verbose:
        print(f"开始创建地理聚类 (每个城市 {n_clusters} 个簇)...")

    required_cols = [city_col, lon_col, lat_col]
    if not all(col in df_processed.columns for col in required_cols):
        if verbose:
            print(f"  错误: DataFrame 缺少必需的列: {required_cols}。跳过聚类...")
        return df_processed

    coord_nan_mask = df_processed[[lon_col, lat_col]].isnull().any(axis=1)
    cluster_col_temp = '地理聚类_temp'
    df_processed[cluster_col_temp] = -1 # 未分配
    all_city_labels = df_processed[city_col].unique()
    
    train_mask = df_processed.index < n_train
    test_mask = df_processed.index >= n_train

    for city_label in all_city_labels:
        city_mask = (df_processed[city_col] == city_label) & (~coord_nan_mask)
        city_data_train = df_processed.loc[city_mask & train_mask, [lon_col, lat_col]]
        city_data_test = df_processed.loc[city_mask & test_mask, [lon_col, lat_col]]

        if len(city_data_train) >= n_clusters:
            try:
                kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
                kmeans.fit(city_data_train)
                
                if not city_data_train.empty:
                    clusters_train = kmeans.predict(city_data_train)
                    df_processed.loc[city_data_train.index, cluster_col_temp] = clusters_train
                if not city_data_test.empty:
                    clusters_test = kmeans.predict(city_data_test)
                    df_processed.loc[city_data_test.index, cluster_col_temp] = clusters_test
            except Exception as e:
                if verbose:
                    print(f"    警告: 城市 {city_label} KMeans 失败: {e}")
        elif len(city_data_train) > 0 and verbose:
            print(f"    警告: 城市 {city_label} 训练集数据点不足 ({len(city_data_train)})，跳过聚类。")

    combined_label_col = '地理聚类_带城市'
    df_processed[combined_label_col] = np.where(
         (df_processed[cluster_col_temp] != -1) & df_processed[city_col].notna(),
         'C' + df_processed[city_col].astype(str).str.split('.').str[0] + '_' + df_processed[cluster_col_temp].astype(str),
         'GeoCluster_Unknown'
    )
    df_processed = pd.get_dummies(df_processed, columns=[combined_label_col], prefix='GeoCluster', drop_first=False)
    df_processed = df_processed.drop(columns=[cluster_col_temp], errors='ignore')
    if verbose:
        print("地理聚类特征创建完毕。")
    return df_processed

In [25]:
# 步骤 1: 数据加载 (Price) 
print("--- 步骤 1: 数据加载 (仅 Price) ---")
try:
    # 您已经加载了训练集，现在加载测试集
    df_test_price_raw = pd.read_csv("data/ruc_Class25Q2_test_price.csv")
    print(f"Price 训练数据已加载: {df_price.shape}")
    print(f"Price 测试数据加载成功: {df_test_price_raw.shape}")
except FileNotFoundError:
    print("错误：未找到 Price 测试数据文件。请确保文件在 'data/' 目录下。")
    exit()

# --- 步骤 2: 存储原始信息与分离 ---
print("\n--- 步骤 2: 存储原始信息与分离 ---")
n_train_price = df_price.shape[0]  # 您已经加载的训练集行数
print(f"Price 训练集原始行数: {n_train_price}")

# 分离目标变量 (Price) - 从您已经加载的df_price中
if 'Price' in df_price.columns:
    y_train_price = df_price['Price'].copy()
    # 对目标变量进行对数变换 
    y_train_ln_price = np.log1p(y_train_price)
    print(f"已分离 Price 目标变量 (y_train_price, y_train_ln_price)，长度: {len(y_train_price)}")
else:
    print("警告: 'Price' 列在训练集中未找到。")
    y_train_price = None
    y_train_ln_price = None

# 分离测试集 ID
if 'ID' in df_test_price_raw.columns:
    test_ids_price = df_test_price_raw['ID'].copy()
    print(f"已分离 Price 测试集 ID (test_ids_price)，长度: {len(test_ids_price)}")
else:
    print("警告: 'ID' 列在测试集中未找到。")
    test_ids_price = None

# --- 步骤 3: 合并数据集以便统一处理 ---
print("\n--- 步骤 3: 合并 Price 训练集与测试集 ---")
# 从训练集中移除 Price 列，从测试集中移除 ID 列
df_train_to_concat = df_price.drop(columns=['Price'], errors='ignore')
df_test_to_concat = df_test_price_raw.drop(columns=['ID'], errors='ignore')

# 添加来源标识
df_train_to_concat['source'] = 'train'
df_test_to_concat['source'] = 'test'

# 合并
df_price_combined = pd.concat([df_train_to_concat, df_test_to_concat], ignore_index=True)
print(f"Price 数据集合并完成。合并后 df_price_combined 形状: {df_price_combined.shape}")

# 清理原始数据框以释放内存
del df_test_price_raw, df_train_to_concat, df_test_to_concat
import gc
gc.collect()

--- 步骤 1: 数据加载 (仅 Price) ---


Price 训练数据已加载: (96048, 55)
Price 测试数据加载成功: (34017, 55)

--- 步骤 2: 存储原始信息与分离 ---
Price 训练集原始行数: 96048
已分离 Price 目标变量 (y_train_price, y_train_ln_price)，长度: 96048
已分离 Price 测试集 ID (test_ids_price)，长度: 34017

--- 步骤 3: 合并 Price 训练集与测试集 ---
Price 数据集合并完成。合并后 df_price_combined 形状: (130065, 55)


C:\Users\lenovo\AppData\Local\Temp\ipykernel_19996\2206414308.py:5: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test_price_raw = pd.read_csv("data/ruc_Class25Q2_test_price.csv")


20

In [26]:

# ============================================================================
# 2. 目标编码函数 (您提供的标准代码)
# ============================================================================

def apply_target_encoding_combined(df, y_target, n_train, loc_cols, n_splits=6, random_state=111):
    """在合并的数据框上对 loc_cols 进行 K-Fold 目标编码。"""
    df_te = df.copy()
    print(f"处理 {loc_cols} (K-Fold Target Encoding, n_splits={n_splits})...")
    if not loc_cols: return df_te 

    existing_loc_cols = [col for col in loc_cols if col in df_te.columns]
    if not existing_loc_cols: return df_te
    print(f"  将对以下存在的列进行编码: {existing_loc_cols}")

    new_col = 'Location_Target_Encoded'
    global_mean = y_target.mean()

    def create_key(df_slice):
        return df_slice[existing_loc_cols].astype(str).agg('_'.join, axis=1)
    df_te['key'] = create_key(df_te)

    X_train_part = df_te.iloc[:n_train].copy()
    X_train_part['target'] = y_target
    full_train_map = X_train_part.groupby('key')['target'].mean()
    print(f"  计算了基于 {n_train} 训练样本的完整均值图谱。")

    X_test_part = df_te.iloc[n_train:].copy()
    df_te.loc[X_test_part.index, new_col] = X_test_part['key'].map(full_train_map).fillna(global_mean)
    print(f"  已将完整图谱应用于 {len(X_test_part)} 个测试样本。")

    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    df_te.loc[X_train_part.index, new_col] = np.nan

    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train_part)):
        X_train_fold = X_train_part.iloc[train_idx]
        X_val_fold = X_train_part.iloc[val_idx]
        fold_map = X_train_fold.groupby('key')['target'].mean()
        df_te.loc[X_val_fold.index, new_col] = X_val_fold['key'].map(fold_map)

    
    is_train_mask = df_te.index < n_train
    is_nan_mask = df_te[new_col].isnull()
    fill_mask = is_train_mask & is_nan_mask
    fill_count = fill_mask.sum()
    if fill_count > 0:
        print(f"    填充训练集中 K-Fold 后剩余的 {fill_count} 个 NaN...")
        values_to_fill = df_te.loc[fill_mask, 'key'].map(full_train_map).values
        df_te.loc[fill_mask, new_col] = values_to_fill

    df_te[new_col] = df_te[new_col].fillna(global_mean)
    cols_to_drop = existing_loc_cols + ['key', 'target']
    df_te = df_te.drop(columns=cols_to_drop, errors='ignore')
    print(f"  K-Fold Target Encoding 完成。新特征: '{new_col}'。")
    return df_te

In [27]:
def preprocess_price(
    df_price: pd.DataFrame,
    X: Optional[pd.DataFrame] = None,
    y_target: Optional[pd.Series] = None,  # 新增：目标变量
    current_year: int = 2025,
    verbose: bool = True,
    create_interactions: bool = True,
    n_train: Optional[int] = None,
    create_geo_features: bool = True,
    apply_target_encoding: bool = True,  # 新增：是否应用目标编码
    location_cols_to_encode: List[str] = None,  # 新增：目标编码的列
) -> pd.DataFrame:
    """
    对楼盘/二手房数据做统一的预处理 & 特征工程，包含丰富的特征变换。
    
    参数:
      - df_price: 原始 pandas.DataFrame
      - X: 初始特征表
      - y_target: 目标变量Series (用于目标编码和特征选择)
      - current_year: 用于计算建筑年龄的当前年份
      - verbose: 是否打印中间信息
      - create_interactions: 是否创建交互特征
      - n_train: 训练集大小，用于地理聚类
      - create_geo_features: 是否创建地理空间特征
      - apply_target_encoding: 是否应用目标编码
      - location_cols_to_encode: 目标编码的地理位置列
    """
    df_price = df_price.copy()
    if X is None:
        X = pd.DataFrame(index=df_price.index)
        
    # 默认删除列
    drop_cols = [
        '区域','板块_comm','环线位置',
        '年份',
        '房屋优势','核心卖点','户型介绍','周边配套','交通出行','客户反馈',
        '产权描述','房屋用途','coord_x','coord_y','抵押信息',
        '建筑结构_comm','套内面积', '配备电梯',
        '物业公司','供热费','物业办公电话','开发商','停车费用',
    ]

    # 1) 删除无关列
    cols_to_drop = [c for c in drop_cols if c in df_price.columns]
    if cols_to_drop:
        df_price = df_price.drop(columns=cols_to_drop)
        if verbose: 
            print("删除列：", cols_to_drop)
    if verbose: 
        print("删除后数据维度:", df_price.shape)

    # 统计空值
    if verbose:
        print("\n缺失值统计（前20）：")
        print(df_price.isnull().sum().sort_values(ascending=False).head(20))

    # 处理区县列
    df_price = handle_district(df_price)
    if '区县' in df_price.columns:
        X['区县'] = df_price['区县']
    
    # 处理板块列
    df_price = handle_plate(df_price)
    if '板块' in df_price.columns:
        X['板块'] = df_price['板块']

    # 调用各列处理函数
    X = price_process_city(df_price, X, verbose)
    X = price_process_ring(df_price, X, verbose)
    X, df_price = price_process_room_layout(df_price, X, verbose)
    X = price_process_floor(df_price, X, verbose)
    X = price_process_area(df_price, X, verbose)
    X = price_process_direction(df_price, X, verbose)
    X = price_process_structure(df_price, X, verbose)
    X = price_process_decoration(df_price, X, verbose)
    X = price_process_elevator_ratio(df_price, X, verbose)
    X = price_process_villa_type(df_price, X, verbose)
    X = price_process_house_age_limit(df_price, X, verbose)
    X = price_process_transaction_type(df_price, X, verbose)
    X = price_process_property_ownership(df_price, X, verbose)
    X = price_process_building_age(df_price, X, current_year, verbose)
    X = price_process_building_counts(df_price, X, verbose)
    X = price_process_green_rate(df_price, X, verbose)
    X = price_process_floor_area_ratio(df_price, X, verbose)
    X = price_process_property_fee(df_price, X, verbose)
    X = price_process_water_supply(df_price, X, verbose)
    X = price_process_heating(df_price, X, verbose)
    X = price_process_power_supply(df_price, X, verbose)
    X = price_process_gas_fee(df_price, X, verbose)
    X = price_process_parking(df_price, X, verbose)

    # 创建地理空间特征
    if create_geo_features and n_train is not None:
        # 将df_price中的地理位置列复制到X中，以便地理处理函数使用
        geo_cols = ['城市', 'lon', 'lat']
        for col in geo_cols:
            if col in df_price.columns and col not in X.columns:
                X[col] = df_price[col]
        
        # 应用地理处理函数
        X = compute_city_center_and_distances(X, n_train, verbose)
        X = create_geo_clusters(X, n_train=n_train, n_clusters=5, verbose=verbose)
        
        # 清理中间的地理位置列
        for col in geo_cols:
            if col in X.columns:
                X = X.drop(columns=[col])

    # 应用目标编码
    if apply_target_encoding and y_target is not None and n_train is not None:
        if location_cols_to_encode is None:
            location_cols_to_encode = ['城市', '区域', '区县', '板块']
        
        # 确保目标编码的列存在于X中
        existing_location_cols = [col for col in location_cols_to_encode if col in X.columns]
        if existing_location_cols:
            if verbose:
                print(f"\n应用目标编码到列: {existing_location_cols}")
            X = apply_target_encoding_combined(X, y_target, n_train, existing_location_cols)
        elif verbose:
            print(f"\n警告: 目标编码列 {location_cols_to_encode} 在特征矩阵中不存在")

    # 创建交互特征
    if create_interactions:
        X = create_interaction_features(X, verbose)

    # 处理缺失值
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if X[col].isna().any():
            median_val = X[col].median()
            X[col] = X[col].fillna(median_val)
            if verbose:
                print(f"填充数值列 '{col}' 的缺失值: {median_val}")


    
    
    # 最终信息展示
    if verbose:
        print("\n=== 特征工程完成 ===")
        print(f"最终 X 列数：{len(X.columns)}")
        print(f"数值型特征: {len(X.select_dtypes(include=[np.number]).columns)}")
        print(f"布尔型特征: {len(X.select_dtypes(include=[bool]).columns)}")
        print(f"对象型特征: {len(X.select_dtypes(include=['object']).columns)}")

    return X  # 返回特征掩码以便后续使用

In [28]:
# ============================================================================
# 2. 评估函数定义
# ============================================================================

def evaluate_model(model, model_name, X_train, X_test, 
                  y_train, y_test, y_train_log, y_test_log,
                  cv_folds=6):
    """
    评估模型在原始房价水平上的性能
    """
    # 训练模型（在对数空间）
    model.fit(X_train, y_train_log)
    
    # 预测（在对数空间）
    y_train_pred_log = model.predict(X_train)
    y_test_pred_log = model.predict(X_test)
    
    # 转换回原始房价水平
    y_train_pred = np.exp(y_train_pred_log)
    y_test_pred = np.exp(y_test_pred_log)
    
    # 计算MAE（原始房价水平）
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    # 计算RMAE（相对平均绝对误差）
    train_rmae = train_mae / np.mean(y_train) * 100
    test_rmae = test_mae / np.mean(y_test) * 100
    
    # 计算RMSE（原始房价水平）
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    # 6折交叉验证（在对数空间计算，但转换为原始房价的MAE）
    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
    cv_scores_mae = []
    cv_scores_rmse = []
    cv_scores_rmae = []
    
    for train_idx, val_idx in kf.split(X_train):
        X_cv_train, X_cv_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_cv_train_log, y_cv_val_log = y_train_log.iloc[train_idx], y_train_log.iloc[val_idx]
        y_cv_val = y_train.iloc[val_idx]
        
        # 在交叉验证的折叠上训练和预测
        model_cv = model.__class__(**model.get_params())
        model_cv.fit(X_cv_train, y_cv_train_log)
        y_cv_pred_log = model_cv.predict(X_cv_val)
        y_cv_pred = np.exp(y_cv_pred_log)
        
        # 计算原始房价的MAE、RMAE和RMSE
        cv_mae = mean_absolute_error(y_cv_val, y_cv_pred)
        cv_rmae = cv_mae / np.mean(y_cv_val) * 100
        cv_rmse = np.sqrt(mean_squared_error(y_cv_val, y_cv_pred))
        
        cv_scores_mae.append(cv_mae)
        cv_scores_rmae.append(cv_rmae)
        cv_scores_rmse.append(cv_rmse)
    
    cv_mae_mean = np.mean(cv_scores_mae)
    cv_mae_std = np.std(cv_scores_mae)
    cv_rmae_mean = np.mean(cv_scores_rmae)
    cv_rmse_mean = np.mean(cv_scores_rmse)
    
    # R²分数（原始房价水平）
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    results = {
        'model_name': model_name,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_rmae': train_rmae,
        'test_rmae': test_rmae,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'cv_mae_mean': cv_mae_mean,
        'cv_mae_std': cv_mae_std,
        'cv_rmae_mean': cv_rmae_mean,
        'cv_rmse_mean': cv_rmse_mean,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'model': model,
        'y_test_pred': y_test_pred
    }
    
    return results

def print_model_results(results):
    """
    打印模型结果
    """
    print(f"\n{results['model_name']} 性能报告:")
    print(f"  样本内 MAE: {results['train_mae']:,.2f}")
    print(f"  样本外 MAE: {results['test_mae']:,.2f}")
    print(f"  6折交叉验证 MAE: {results['cv_mae_mean']:,.2f} (±{results['cv_mae_std']:,.2f})")
    print(f"  样本内 RMAE: {results['train_rmae']:.2f}%")
    print(f"  样本外 RMAE: {results['test_rmae']:.2f}%")
    print(f"  6折交叉验证 RMAE: {results['cv_rmae_mean']:.2f}%")
    print(f"  样本内 RMSE: {results['train_rmse']:,.2f}")
    print(f"  样本外 RMSE: {results['test_rmse']:,.2f}")
    print(f"  6折交叉验证 RMSE: {results['cv_rmse_mean']:,.2f}")
    print(f"  样本内 R²: {results['train_r2']:.4f}")
    print(f"  样本外 R²: {results['test_r2']:.4f}")

In [29]:
#============================================================================
# 模型训练和评估模块
# ============================================================================

def train_and_evaluate_models(X_train, X_test, y_train, y_test, y_train_log, y_test_log, verbose=True):
    """
    训练和评估四个线性模型：OLS、LASSO、Ridge、ElasticNet
    
    返回:
    - all_results: 所有模型的结果字典
    - best_model_info: 最佳模型信息
    - models_dict: 训练好的模型字典
    """
    
    from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
    from sklearn.model_selection import GridSearchCV
    
    all_results = {}
    models_dict = {}
    
    # ============================================================================
    # 1. OLS 线性回归
    # ============================================================================
    
    if verbose:
        print("\n" + "="*60)
        print("OLS 线性回归")
        print("="*60)
    
    # 创建并训练OLS模型
    ols_model = LinearRegression()
    
    # 评估OLS模型
    ols_results = evaluate_model(
        ols_model, "OLS", X_train, X_test, y_train, y_test, 
        y_train_log, y_test_log
    )
    
    if verbose:
        print_model_results(ols_results)
    
    # OLS特征重要性分析
    if hasattr(ols_model, 'coef_'):
        ols_coefficients = pd.DataFrame({
            'feature': X_train.columns,
            'coefficient': ols_model.coef_,
            'abs_coefficient': np.abs(ols_model.coef_)
        }).sort_values('abs_coefficient', ascending=False)
        
        if verbose:
            print(f"\nOLS 前10个最重要特征:")
            print("-" * 50)
            for idx, row in ols_coefficients.head(10).iterrows():
                sign = "↓" if row['coefficient'] < 0 else "↑"
                print(f"  {row['feature']:<30} {row['coefficient']:>8.4f} {sign}")
    
    all_results["OLS"] = ols_results
    models_dict["OLS"] = ols_model
    
    # ============================================================================
    # 2. LASSO 回归（带超参数调优）
    # ============================================================================
    
    if verbose:
        print("\n" + "="*60)
        print("LASSO 回归 - 超参数调优")
        print("="*60)
    
    # 定义参数网格
    lasso_param_grid = {
        'alpha': [0.001, 0.01, 0.1, 1, 10, 100],
        'max_iter': [5000, 10000],
        'selection': ['cyclic', 'random']
    }
    
    if verbose:
        print("正在进行LASSO超参数搜索...")
    
    lasso_grid = GridSearchCV(
        Lasso(random_state=42),
        lasso_param_grid,
        cv=5,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1 if verbose else 0
    )
    
    # 在对数空间进行网格搜索
    lasso_grid.fit(X_train, y_train_log)
    
    if verbose:
        print(f"LASSO最佳参数: {lasso_grid.best_params_}")
        print(f"LASSO最佳交叉验证分数: {-lasso_grid.best_score_:.4f}")
    
    # 使用最佳参数训练LASSO模型
    best_lasso = lasso_grid.best_estimator_
    lasso_results = evaluate_model(
        best_lasso, "LASSO", X_train, X_test, y_train, y_test, 
        y_train_log, y_test_log
    )
    
    if verbose:
        print_model_results(lasso_results)
    
    # LASSO特征重要性分析（稀疏性）
    lasso_coefficients = pd.DataFrame({
        'feature': X_train.columns,
        'coefficient': best_lasso.coef_,
        'abs_coefficient': np.abs(best_lasso.coef_)
    }).sort_values('abs_coefficient', ascending=False)
    
    non_zero_features = lasso_coefficients[lasso_coefficients['coefficient'] != 0]
    
    if verbose:
        print(f"\nLASSO 选择了 {len(non_zero_features)} 个非零特征（共 {len(lasso_coefficients)} 个）")
        print(f"\nLASSO 前10个最重要特征:")
        print("-" * 50)
        for idx, row in lasso_coefficients.head(10).iterrows():
            sign = "↓" if row['coefficient'] < 0 else "↑"
            print(f"  {row['feature']:<30} {row['coefficient']:>8.4f} {sign}")
    
    all_results["LASSO"] = lasso_results
    models_dict["LASSO"] = best_lasso
    
    # ============================================================================
    # 3. Ridge 回归（带超参数调优）
    # ============================================================================
    
    if verbose:
        print("\n" + "="*60)
        print("Ridge 回归 - 超参数调优")
        print("="*60)
    
    # 定义参数网格
    ridge_param_grid = {
        'alpha': [0.1, 1, 10],
        'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga']
    }
    
    if verbose:
        print("正在进行Ridge超参数搜索...")
    
    ridge_grid = GridSearchCV(
        Ridge(random_state=42),
        ridge_param_grid,
        cv=5,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1 if verbose else 0
    )
    
    # 在对数空间进行网格搜索
    ridge_grid.fit(X_train, y_train_log)
    
    if verbose:
        print(f"Ridge最佳参数: {ridge_grid.best_params_}")
        print(f"Ridge最佳交叉验证分数: {-ridge_grid.best_score_:.4f}")
    
    # 使用最佳参数训练Ridge模型
    best_ridge = ridge_grid.best_estimator_
    ridge_results = evaluate_model(
        best_ridge, "Ridge", X_train, X_test, y_train, y_test, 
        y_train_log, y_test_log
    )
    
    if verbose:
        print_model_results(ridge_results)
    
    # Ridge特征重要性分析
    ridge_coefficients = pd.DataFrame({
        'feature': X_train.columns,
        'coefficient': best_ridge.coef_,
        'abs_coefficient': np.abs(best_ridge.coef_)
    }).sort_values('abs_coefficient', ascending=False)
    
    if verbose:
        print(f"\nRidge 前10个最重要特征:")
        print("-" * 50)
        for idx, row in ridge_coefficients.head(10).iterrows():
            sign = "↓" if row['coefficient'] < 0 else "↑"
            print(f"  {row['feature']:<30} {row['coefficient']:>8.4f} {sign}")
    
    all_results["Ridge"] = ridge_results
    models_dict["Ridge"] = best_ridge
    
    # ============================================================================
    # 4. ElasticNet 回归（带超参数调优）
    # ============================================================================
    
    if verbose:
        print("\n" + "="*60)
        print("ElasticNet 回归 - 超参数调优")
        print("="*60)
    
    # 定义参数网格
    elasticnet_param_grid = {
        'alpha': [0.001, 0.01, 0.1],
        'l1_ratio': [0.1, 0.3, 0.5],
        'max_iter': [5000]
    }
    
    if verbose:
        print("正在进行ElasticNet超参数搜索...")
    
    elasticnet_grid = GridSearchCV(
        ElasticNet(random_state=42),
        elasticnet_param_grid,
        cv=5,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1 if verbose else 0
    )
    
    # 在对数空间进行网格搜索
    elasticnet_grid.fit(X_train, y_train_log)
    
    if verbose:
        print(f"ElasticNet最佳参数: {elasticnet_grid.best_params_}")
        print(f"ElasticNet最佳交叉验证分数: {-elasticnet_grid.best_score_:.4f}")
    
    # 使用最佳参数训练ElasticNet模型
    best_elasticnet = elasticnet_grid.best_estimator_
    elasticnet_results = evaluate_model(
        best_elasticnet, "ElasticNet", X_train, X_test, y_train, y_test, 
        y_train_log, y_test_log
    )
    
    if verbose:
        print_model_results(elasticnet_results)
    
    # ElasticNet特征重要性分析
    elasticnet_coefficients = pd.DataFrame({
        'feature': X_train.columns,
        'coefficient': best_elasticnet.coef_,
        'abs_coefficient': np.abs(best_elasticnet.coef_)
    }).sort_values('abs_coefficient', ascending=False)
    
    non_zero_features_en = elasticnet_coefficients[elasticnet_coefficients['coefficient'] != 0]
    
    if verbose:
        print(f"\nElasticNet 选择了 {len(non_zero_features_en)} 个非零特征（共 {len(elasticnet_coefficients)} 个）")
        print(f"\nElasticNet 前10个最重要特征:")
        print("-" * 50)
        for idx, row in elasticnet_coefficients.head(10).iterrows():
            sign = "↓" if row['coefficient'] < 0 else "↑"
            print(f"  {row['feature']:<30} {row['coefficient']:>8.4f} {sign}")
    
    all_results["ElasticNet"] = elasticnet_results
    models_dict["ElasticNet"] = best_elasticnet
    
    # ============================================================================
    # 模型比较和选择
    # ============================================================================
    
    if verbose:
        print("\n" + "="*80)
        print("模型比较总结")
        print("="*80)
    
    # 选择最佳模型（基于测试集MAE）
    best_model_name = None
    best_test_mae = float('inf')
    
    for model_name, results in all_results.items():
        test_mae = results['test_mae']
        if verbose:
            print(f"{model_name:<12}: 测试集MAE = {test_mae:,.2f}, RMAE = {results['test_rmae']:.2f}%")
        
        if test_mae < best_test_mae:
            best_test_mae = test_mae
            best_model_name = model_name
    
    best_model_info = all_results[best_model_name]
    
    if verbose:
        print(f"\n🎯 最佳模型: {best_model_name}")
        print(f"最佳测试集MAE: {best_test_mae:,.2f}")
        print(f"最佳测试集RMAE: {best_model_info['test_rmae']:.2f}%")
    
    return all_results, best_model_info, models_dict

In [30]:
def detect_and_remove_outliers(X: pd.DataFrame, y: pd.Series = None, 
                              method: str = 'quantile', 
                              threshold: float = 0.99,
                              verbose: bool = True) -> Tuple[pd.DataFrame, pd.Series, dict]:
    """
    检测并删除异常值
    
    参数:
    - X: 特征DataFrame
    - y: 目标变量Series (可选)
    - method: 异常值检测方法 ('quantile', 'zscore', 'iqr')
    - threshold: 阈值 (对于quantile是分位数, 对于zscore是标准差倍数)
    - verbose: 是否打印详细信息
    
    返回:
    - X_clean: 清理后的特征
    - y_clean: 清理后的目标变量 (如果提供了y)
    - outlier_info: 异常值统计信息
    """
    
    if verbose:
        print("=" * 50)
        print("异常值检测与处理")
        print("=" * 50)
    
    # 选择数值型列进行异常值检测
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    bool_cols = X.select_dtypes(include=[bool]).columns.tolist()
    
    if verbose:
        print(f"检测数值型列 ({len(numeric_cols)}个): {numeric_cols}")
        print(f"跳过布尔型列 ({len(bool_cols)}个)")
    
    outlier_info = {
        'method': method,
        'threshold': threshold,
        'columns_analyzed': numeric_cols,
        'outliers_by_column': {},
        'total_outlier_rows': set(),
        'removed_rows_count': 0
    }
    
    # 方法1: 分位数方法 (默认)
    if method == 'quantile':
        for col in numeric_cols:
            lower_bound = X[col].quantile(1 - threshold)
            upper_bound = X[col].quantile(threshold)
            
            outliers = X[(X[col] < lower_bound) | (X[col] > upper_bound)].index
            outlier_info['outliers_by_column'][col] = {
                'count': len(outliers),
                'percentage': len(outliers) / len(X) * 100,
                'lower_bound': lower_bound,
                'upper_bound': upper_bound,
                'min': X[col].min(),
                'max': X[col].max()
            }
            outlier_info['total_outlier_rows'].update(outliers)
    
    # 方法2: Z-score方法
    elif method == 'zscore':
        for col in numeric_cols:
            z_scores = np.abs(stats.zscore(X[col].fillna(X[col].median())))
            outliers = X[z_scores > threshold].index
            
            outlier_info['outliers_by_column'][col] = {
                'count': len(outliers),
                'percentage': len(outliers) / len(X) * 100,
                'threshold_z': threshold,
                'min_z': z_scores.min(),
                'max_z': z_scores.max()
            }
            outlier_info['total_outlier_rows'].update(outliers)
    
    # 方法3: IQR方法
    elif method == 'iqr':
        for col in numeric_cols:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR
            
            outliers = X[(X[col] < lower_bound) | (X[col] > upper_bound)].index
            outlier_info['outliers_by_column'][col] = {
                'count': len(outliers),
                'percentage': len(outliers) / len(X) * 100,
                'lower_bound': lower_bound,
                'upper_bound': upper_bound,
                'IQR': IQR
            }
            outlier_info['total_outlier_rows'].update(outliers)
    
    # 转换为有序列表
    outlier_info['total_outlier_rows'] = list(outlier_info['total_outlier_rows'])
    outlier_info['removed_rows_count'] = len(outlier_info['total_outlier_rows'])
    
    # 删除异常值行
    if verbose:
        print(f"\n检测到异常值行数: {outlier_info['removed_rows_count']}")
        print(f"占总数据比例: {outlier_info['removed_rows_count'] / len(X) * 100:.2f}%")
    
    # 创建清理后的数据
    clean_mask = ~X.index.isin(outlier_info['total_outlier_rows'])
    X_clean = X[clean_mask].copy()
    
    if y is not None:
        y_clean = y[clean_mask].copy()
    else:
        y_clean = None
    
    if verbose:
        print(f"清理后数据形状: {X_clean.shape}")
        if y is not None:
            print(f"清理后目标变量形状: {y_clean.shape}")
    
    return X_clean, y_clean, outlier_info

In [31]:
def prepare_model_data(X_clean, y_clean, test_size=0.2, random_state=42):
    """
    准备建模数据：对数化目标变量，划分训练测试集
    """
    print("数据准备阶段")
    print("-" * 50)
    
    # 对目标变量进行对数化
    y_log = np.log(y_clean)
    
    print(f"目标变量统计:")
    print(f"  原始范围: [{y_clean.min():.2f}, {y_clean.max():.2f}]")
    print(f"  原始均值: {y_clean.mean():.2f}, 标准差: {y_clean.std():.2f}")
    print(f"  对数化后: [{y_log.min():.2f}, {y_log.max():.2f}]")
    
    # 划分训练测试集
    X_train, X_val, y_train, y_val = train_test_split(
        X_clean, y_clean, test_size=test_size, random_state=random_state
    )
    _, _, y_train_log, y_val_log = train_test_split(
        X_clean, y_log, test_size=test_size, random_state=random_state
    )
    
    print(f"\n数据划分:")
    print(f"  训练集: {X_train.shape[0]} 样本")
    print(f"  验证集: {X_val.shape[0]} 样本")
    print(f"  特征数: {X_train.shape[1]}")
    
    return X_train, X_val, y_train, y_val, y_train_log, y_val_log

In [32]:
# ============================================================================
# 完整的训练和预测流程（整合版本）- 为四个模型分别生成预测结果
# ============================================================================

def complete_training_pipeline(
    df_combined: pd.DataFrame,
    y_train_target: pd.Series,
    n_train: int,
    test_ids: pd.Series,
    current_year: int = 2025,
    apply_outlier_detection: bool = False,
    outlier_threshold: float = 0.99,
    test_size: float = 0.2,
    random_state: int = 42,
    verbose: bool = True
):
    """
    完整的房价预测流程：特征工程 → 异常值处理 → 模型训练 → 预测
    为四个模型（OLS、LASSO、Ridge、ElasticNet）分别生成预测结果
    """
    
    print("=" * 80)
    print("开始完整房价预测流程 - 四个模型分别预测")
    print("=" * 80)
    
    # 步骤1: 特征工程
    print("\n--- 步骤1: 特征工程 ---")
    X_processed = preprocess_price(
        df_price=df_combined,
        y_target=y_train_target,
        n_train=n_train,
        current_year=current_year,
        create_geo_features=True,
        apply_target_encoding=True,
        location_cols_to_encode=['城市', '区县', '板块'],
        verbose=verbose
    )
    
    print(f"特征工程完成，特征矩阵形状: {X_processed.shape}")
    
    # 步骤2: 分离训练集和测试集
    print("\n--- 步骤2: 分离训练集和测试集 ---")
    X_train_full = X_processed.iloc[:n_train].copy()
    X_test_final = X_processed.iloc[n_train:].copy()
    
    y_train_full = y_train_target.copy()
    y_train_log = np.log(y_train_full)
    
    print(f"训练集: {X_train_full.shape}")
    print(f"测试集: {X_test_final.shape}")
    
    # 步骤3: 异常值处理
    if apply_outlier_detection:
        print("\n--- 步骤3: 异常值检测与处理 ---")
        X_clean, y_clean, outlier_info = detect_and_remove_outliers(
            X_train_full, y_train_full, 
            method='quantile', 
            threshold=outlier_threshold,
            verbose=verbose
        )
        y_clean_log = np.log(y_clean)
    else:
        print("\n--- 步骤3: 跳过异常值检测 ---")
        X_clean, y_clean = X_train_full.copy(), y_train_full.copy()
        y_clean_log = y_train_log.copy()
        outlier_info = {}
    
    print(f"清理后训练集: {X_clean.shape}")
    
    # 步骤4: 数据准备和标准化
    print("\n--- 步骤4: 数据准备和标准化 ---")
    
    # 4.1 划分训练测试集
    X_train, X_val, y_train, y_val, y_train_log, y_val_log = prepare_model_data(
        X_clean, y_clean, test_size=test_size, random_state=random_state
    )
    
    print(f"训练集: {X_train.shape}, 验证集: {X_val.shape}")
    
    # 4.2 标准化特征
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns
    
    scaler = StandardScaler()
    X_train_scaled = X_train.copy()
    X_val_scaled = X_val.copy()
    X_test_final_scaled = X_test_final.copy()
    
    X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])
    X_test_final_scaled[numeric_cols] = scaler.transform(X_test_final[numeric_cols])
    
    print(f"标准化完成:")
    print(f"  训练集: {X_train_scaled.shape}")
    print(f"  验证集: {X_val_scaled.shape}")
    print(f"  测试集: {X_test_final_scaled.shape}")
    
    # 步骤5: 模型训练和评估
    print("\n--- 步骤5: 模型训练和评估 ---")
    
    all_results, best_model_info, models_dict = train_and_evaluate_models(
        X_train_scaled, X_val_scaled, y_train, y_val, y_train_log, y_val_log, verbose
    )
    
    # 步骤6: 在完整训练集上重新训练所有模型
    print("\n--- 步骤6: 在完整训练集上重新训练所有模型 ---")
    
    # 在完整清理后的数据上重新标准化
    X_full_clean_scaled = X_clean.copy()
    X_full_clean_scaled[numeric_cols] = scaler.transform(X_clean[numeric_cols])
    
    print(f"完整训练集最终形状: {X_full_clean_scaled.shape}")
    
    # 重新训练所有模型
    final_models = {}
    for model_name, model in models_dict.items():
        print(f"重新训练 {model_name} 模型...")
        # 克隆模型以避免修改原始模型
        from sklearn.base import clone
        final_model = clone(model)
        final_model.fit(X_full_clean_scaled, y_clean_log)
        final_models[model_name] = final_model
    
    print("所有模型重新训练完成!")
    
    # 步骤7: 特征重要性分析（使用最佳模型）
    print("\n--- 步骤7: 特征重要性分析 ---")
    
    best_model_name = best_model_info['model_name']
    best_final_model = final_models[best_model_name]
    
    if hasattr(best_final_model, 'coef_'):
        feature_importance = pd.DataFrame({
            'feature': X_full_clean_scaled.columns,
            'coefficient': best_final_model.coef_,
            'abs_coefficient': np.abs(best_final_model.coef_)
        }).sort_values('abs_coefficient', ascending=False)
        
        print(f"\n最佳模型 ({best_model_name}) 前20个最重要特征:")
        print("-" * 60)
        for idx, row in feature_importance.head(20).iterrows():
            sign = "↓" if row['coefficient'] < 0 else "↑"
            print(f"  {row['feature']:<40} {row['coefficient']:>8.4f} {sign}")
    
    # 步骤8: 为所有模型生成预测结果
    print("\n--- 步骤8: 为所有模型生成预测结果 ---")
    
    timestamp = pd.Timestamp.now().strftime("%m%d%H%M")
    output_files = {}
    
    for model_name, model in final_models.items():
        print(f"\n使用 {model_name} 模型进行预测...")
        
        # 进行预测
        y_test_pred_log = model.predict(X_test_final_scaled)
        y_test_pred = np.exp(y_test_pred_log)
        
        # 生成输出文件
        output = pd.DataFrame({
            "ID": test_ids,
            "Price": y_test_pred
        })
        
        # 保存预测结果
        output_file = f"data/final_predicted_price_{model_name}_{timestamp}.csv"
        output.to_csv(output_file, index=False, encoding="utf-8-sig")
        output_files[model_name] = output_file
        
        print(f"✅ {model_name} 预测完成！结果已保存至 {output_file}")
        print(f"{model_name} 预测价格统计:")
        print(f"  最小值: {y_test_pred.min():,.2f}")
        print(f"  最大值: {y_test_pred.max():,.2f}")
        print(f"  平均值: {y_test_pred.mean():,.2f}")
        print(f"  中位数: {np.median(y_test_pred):,.2f}")
        
        # 只显示最佳模型的预测结果预览
        if model_name == best_model_name:
            print(f"\n最佳模型 {model_name} 预测结果预览:")
            print(output.head(10))
    
    # 步骤9: 模型性能比较
    print("\n--- 步骤9: 模型性能比较 ---")
    
    print(f"\n所有模型在验证集上的性能比较:")
    print("-" * 80)
    print(f"{'模型':<12} {'测试集MAE':<12} {'测试集RMAE':<12} {'测试集R²':<10}")
    print("-" * 80)
    
    for model_name, results in all_results.items():
        print(f"{model_name:<12} {results['test_mae']:>10,.2f} {results['test_rmae']:>10.2f}% {results['test_r2']:>9.4f}")
    
    print(f"\n🎯 最佳模型: {best_model_name}")
    print(f"最佳测试集MAE: {best_model_info['test_mae']:,.2f}")
    print(f"最佳测试集RMAE: {best_model_info['test_rmae']:.2f}%")
    
    # 返回所有重要信息
    return {
        'final_models': final_models,
        'scaler': scaler,
        'best_model_info': best_model_info,
        'all_results': all_results,
        'models_dict': models_dict,
        'output_files': output_files,
        'feature_importance': feature_importance if 'feature_importance' in locals() else None,
        'outlier_info': outlier_info,
        'X_processed_shape': X_processed.shape,
    }

In [33]:
result_modified = complete_training_pipeline(
    df_combined=df_price_combined,  # 您合并的数据
    y_train_target=y_train_price,    # 训练集目标变量
    n_train=n_train_price,           # 训练集大小
    test_ids=test_ids_price,         # 测试集ID
    current_year=2025,
    apply_outlier_detection=True,
    outlier_threshold=0.99,
    test_size=0.2,
    random_state=42,
    verbose=True
)

print(f"\n🎉 完整流程执行完毕！")
print(f"最佳模型: {result_modified['best_model_info']['model_name']}")
print(f"测试集MAE: {result_modified['best_model_info']['test_mae']:,.2f}")
print(f"测试集RMAE: {result_modified['best_model_info']['test_rmae']:.2f}%")

print(f"\n所有模型的输出文件:")
for model_name, file_path in result_modified['output_files'].items():
    print(f"  {model_name}: {file_path}")

开始完整房价预测流程 - 四个模型分别预测

--- 步骤1: 特征工程 ---
删除列： ['区域', '板块_comm', '环线位置', '年份', '房屋优势', '核心卖点', '户型介绍', '周边配套', '交通出行', '客户反馈', '产权描述', '房屋用途', 'coord_x', 'coord_y', '抵押信息', '建筑结构_comm', '套内面积', '配备电梯', '物业公司', '供热费', '物业办公电话', '开发商', '停车费用']
删除后数据维度: (130065, 32)

缺失值统计（前20）：
别墅类型     128785
供暖        89015
环线        79901
房屋年限      54592
建筑年代      43227
停车位       42838
容 积 率     41310
绿 化 率     40912
燃气费       40661
物 业 费     38541
供水        37404
供电        37380
物业类别      34875
上次交易      32371
区县        10902
房屋总数      10775
楼栋总数      10775
梯户比例       2826
房屋户型        593
建筑结构        593
dtype: int64
处理 [区县]...
  已将 '区县' 填充缺失值并转换为 'string' 类型。
  处理后 '区县' 的唯一值 (示例): ['65.0' '62.0' '81.0' '112.0' '68.0' '7.0' '123.0' '28.0' '11.0' '95.0']
处理 [板块]...
  已将 '板块' 填充缺失值并转换为 'string' 类型。
加入城市独热编码，新增列数： 12
加入环线（中位数填充）
加入房屋户型拆分特征：室/厅/厨/卫 + 比率特征
加入所在楼层特征：总楼层 + 楼层类型独热编码
加入建筑面积特征：数值 + 对数 + 平方
加入朝向特征：八方向 + 综合得分
加入建筑结构独热编码
加入装修情况特征：独热编码 + 等级评分
加入梯户特征：梯数、户数 + 梯户比
加入别墅类型特征：数值编码 + 布尔标志
加入房屋年限编码
加入交易权属布

C:\Users\lenovo\AppData\Local\Temp\ipykernel_19996\3536126342.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_out['距离中心_公里'].fillna(median_dist_train, inplace=True)


  计算了 '距离中心_公里'，并用训练集中位数 (15.30 km) 填充了 NaN。
地理距离特征创建完毕。
开始创建地理聚类 (每个城市 5 个簇)...


d:\APP\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=13.
  warnings.warn(
d:\APP\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=9.
  warnings.warn(
d:\APP\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
d:\APP\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less

地理聚类特征创建完毕。

应用目标编码到列: ['区县', '板块']
处理 ['区县', '板块'] (K-Fold Target Encoding, n_splits=6)...
  将对以下存在的列进行编码: ['区县', '板块']
  计算了基于 96048 训练样本的完整均值图谱。
  已将完整图谱应用于 34017 个测试样本。
    填充训练集中 K-Fold 后剩余的 102 个 NaN...
  K-Fold Target Encoding 完成。新特征: 'Location_Target_Encoded'。
创建交互特征完成

=== 特征工程完成 ===
最终 X 列数：148
数值型特征: 35
布尔型特征: 113
对象型特征: 0
特征工程完成，特征矩阵形状: (130065, 148)

--- 步骤2: 分离训练集和测试集 ---
训练集: (96048, 148)
测试集: (34017, 148)

--- 步骤3: 异常值检测与处理 ---
异常值检测与处理
检测数值型列 (35个): ['环线', '室', '厅', '厨', '卫', '房间总数', '卫室比', '总楼层', '建筑面积', '建筑面积_log', '建筑面积_sq', '朝向得分', '装修等级', '梯数', '户数', '梯户比', '别墅类型', '房屋年限', '建筑年龄', '房屋总数', '楼栋总数', '每栋户数', '每栋户数_log', '绿化率', '容积率', '物业费', '物业费_log', '供暖_encoded', '燃气费', '停车位', '距离中心_公里', '距离中心_公里_平方', 'Location_Target_Encoded', '每室面积', '每厅面积']
跳过布尔型列 (113个)

检测到异常值行数: 22475
占总数据比例: 23.40%
清理后数据形状: (73573, 148)
清理后目标变量形状: (73573,)
清理后训练集: (73573, 148)

--- 步骤4: 数据准备和标准化 ---
数据准备阶段
--------------------------------------------------
目标变量统计:
  原始范围: [122192.52, 5363695.

## rent

In [34]:
# 初始化空的X
X = pd.DataFrame()

In [35]:

def rent_process_city(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理城市列 - 独热编码为布尔类型"""
    if '城市' in df_rent.columns:
        city_dummies = pd.get_dummies(df_rent['城市'], prefix='城市').astype(bool)
        X = pd.concat([X, city_dummies], axis=1)
        if verbose: 
            print(f"加入城市独热编码，新增列数：{city_dummies.shape[1]}")
    return X


In [36]:
def rent_process_room_layout(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理户型列 - 解析室/厅/卫 + 创建比率特征"""
    def parse_room_info(x):
        room, hall, bath = pd.NA, pd.NA, pd.NA
        if not pd.isna(x) and x != '·':
            room_match = re.search(r'(\d+)(室|房间|居室)', str(x))
            hall_match = re.search(r'(\d+)厅', str(x))
            bath_match = re.search(r'(\d+)卫', str(x))
            room = int(room_match.group(1)) if room_match else 0
            hall = int(hall_match.group(1)) if hall_match else 0
            bath = int(bath_match.group(1)) if bath_match else 0
        return pd.Series([room, hall, bath], index=['室', '厅', '卫'], dtype='Int64')
    
    if '户型' in df_rent.columns:
        room_features = df_rent['户型'].apply(parse_room_info)
        
        # 填充缺失值
        for col in ['室', '厅', '卫']:
            median_val = room_features[col].median()
            room_features[col] = room_features[col].fillna(median_val).astype('Int64')
        
        # 基础特征
        X = pd.concat([X, room_features], axis=1)
        
        # 特征工程优化
        # 1. 房间总数
        X['房间总数'] = room_features['室'] + room_features['厅']
        
        # 2. 卫室比（重要特征）
        X['卫室比'] = room_features['卫'] / (room_features['室'] + 0.01)
        
        # 3. 厅室比
        X['厅室比'] = room_features['厅'] / (room_features['室'] + 0.01)
        
        # 4. 是否为一室户
        X['是一室户'] = (room_features['室'] == 1).astype(bool)
        
        if verbose:
            print("加入户型特征：室/厅/卫 + 比率特征 + 一室户标志")
    
    return X


In [37]:
def rent_process_decoration(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理装修列 - 简化处理，只有精装修一个有效值"""
    if '装修' in df_rent.columns:
        # 简化为布尔特征：是否为精装修
        X['是精装修'] = (df_rent['装修'] == '精装修').astype(bool)
        
        # 填充空值为False（非精装修）
        X['是精装修'] = X['是精装修'].fillna(False)
        
        if verbose:
            print("加入装修特征：精装修布尔标志")
            print(f"精装修比例: {X['是精装修'].mean():.2%}")
    
    return X

In [38]:
def rent_process_floor(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理楼层列 - 解析当前楼层/总楼层 + 创建楼层特征"""
    def parse_floor(x):
        if pd.isna(x):
            return [np.nan, np.nan]
        x = str(x).strip()
        if '/' in x:
            current_part, total_part = x.split('/', 1)
            total_match = re.search(r'(\d+)', total_part)
            total = int(total_match.group(1)) if total_match else np.nan
            
            current = np.nan
            if '地下室' in current_part:
                current = 0
            elif '低' in current_part and not np.isnan(total):
                current = int(total * (1/3))
            elif '中' in current_part and not np.isnan(total):
                current = int(total * (1/2))
            elif '高' in current_part and not np.isnan(total):
                current = int(total * (4/5))
            else:
                current_match = re.search(r'(\d+)', current_part)
                current = int(current_match.group(1)) if current_match else np.nan
            return [current, total]
        return [np.nan, np.nan]
    
    if '楼层' in df_rent.columns:
        floor_features = df_rent['楼层'].apply(lambda x: pd.Series(parse_floor(x)))
        floor_features.columns = ['当前楼层', '总楼层']
        
        # 填充缺失值
        floor_features['当前楼层'] = floor_features['当前楼层'].fillna(floor_features['当前楼层'].median())
        floor_features['总楼层'] = floor_features['总楼层'].fillna(floor_features['总楼层'].median())
        
        # 基础特征
        X = pd.concat([X, floor_features], axis=1)
        
        # 特征工程优化
        # 1. 楼层位置比（重要特征）
        X['楼层位置比'] = (floor_features['当前楼层'] - 1) / (floor_features['总楼层'] - 1 + 0.01)
        
        # 2. 是否为低楼层
        X['是低楼层'] = (floor_features['当前楼层'] <= 3).astype(bool)
        
        # 3. 是否为高楼层
        X['是高楼层'] = (floor_features['当前楼层'] >= floor_features['总楼层'] - 2).astype(bool)
        
        # 4. 是否为中层
        X['是中层'] = ((floor_features['当前楼层'] > 3) & 
                      (floor_features['当前楼层'] < floor_features['总楼层'] - 2)).astype(bool)
        
        if verbose:
            print("加入楼层特征：当前/总楼层 + 位置比 + 楼层类型标志")
    
    return X

In [39]:
def rent_process_area(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理面积列 - 数值化 + 对数变换 + 分箱 + 多项式"""
    if '面积' in df_rent.columns:
        # 基础面积特征
        area_series = df_rent['面积'].str.replace('㎡', '', regex=False).astype(float)
        area_filled = area_series.fillna(area_series.median())
        X['面积'] = area_filled
        
        # 特征工程优化
        # 1. 面积对数变换
        X['面积_log'] = safe_log_transform(area_filled)
        
        # 2. 面积平方
        X['面积_sq'] = area_filled ** 2
        
        
        if verbose:
            print("加入面积特征：数值 + 对数 + 平方")
    
    return X

In [40]:
def rent_process_direction(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理朝向列 - 八方向布尔特征 + 创建朝向得分"""
    if '朝向' in df_rent.columns:
        directions = ['东', '南', '西', '北', '东南', '西南', '东北', '西北']
        direction_features = pd.DataFrame(index=df_rent.index)
        
        # 基础方向特征
        for d in directions:
            direction_features[f'朝向_{d}'] = df_rent['朝向'].apply(lambda x: 1 if d in str(x) else 0)
        direction_features = direction_features.astype(bool)
        
        # 创建朝向得分（南=3, 东/东南=2, 北/西=1, 其他=0）
        direction_score = pd.Series(0, index=df_rent.index)
        direction_score += direction_features['朝向_南'] * 3
        direction_score += direction_features['朝向_东南'] * 2
        direction_score += direction_features['朝向_东'] * 2
        direction_score += direction_features['朝向_西南'] * 1
        direction_score += direction_features['朝向_东北'] * 1
        direction_score += direction_features['朝向_北'] * 1
        direction_score += direction_features['朝向_西'] * 1
        
        X['朝向得分'] = direction_score
        X = pd.concat([X, direction_features], axis=1)
        
        # 是否为南北通透
        X['南北通透'] = (direction_features['朝向_南'] & direction_features['朝向_北']).astype(bool)
        
        if verbose:
            print("加入朝向特征：八方向 + 综合得分 + 南北通透标志")
    
    return X

In [41]:
def rent_process_structure(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理建筑结构列 - 拆分为单独的结构类型布尔特征"""
    if '建筑结构' in df_rent.columns:
        # 定义四种主要结构类型
        structure_types = ['塔楼', '板楼', '塔板结合', '平房']
        
        # 为每种结构类型创建布尔特征
        for structure_type in structure_types:
            X[f'结构_{structure_type}'] = df_rent['建筑结构'].str.contains(structure_type, na=False).astype(bool)
        
        # 计算包含的结构类型数量
        X['结构类型数量'] = 0
        for structure_type in structure_types:
            X['结构类型数量'] += X[f'结构_{structure_type}']
        
        if verbose:
            print("加入建筑结构特征：四种结构类型布尔标志 + 结构类型数量")
    
    return X

In [42]:

def rent_process_transaction_time(df_rent: pd.DataFrame, X: pd.DataFrame, current_year: int = 2025, verbose: bool = True) -> pd.DataFrame:
    """处理交易时间列 - 计算距离天数 + 创建时间特征"""
    if '交易时间' in df_rent.columns:
        # 转换为日期格式
        df_rent['交易时间_dt'] = pd.to_datetime(df_rent['交易时间'], errors='coerce')
        today = pd.to_datetime(f'{current_year}-10-25')
        df_rent['交易时间天数'] = (today - df_rent['交易时间_dt']).dt.days
        df_rent['交易时间天数'] = df_rent['交易时间天数'].fillna(-1).astype(int)
        
        # 基础特征
        X['交易时间天数'] = df_rent['交易时间天数']
        
        # 特征工程优化
        # 1. 是否为近期交易（30天内）
        X['是近期交易'] = (df_rent['交易时间天数'] <= 30).astype(bool)
        
        if verbose:
            print("加入交易时间特征：天数 + 近期标志")
    
    return X

In [43]:
def rent_process_payment_method(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理付款方式列 - 有序编码 + 创建特征"""
    if '付款方式' in df_rent.columns:
        # 处理异常值和空值
        abnormal_values = ['https://img.ljcdn.com/usercent', 'https://image1.ljcdn.com/rent-']
        mode_val = '季付价'
        df_rent['付款方式_cleaned'] = df_rent['付款方式'].replace(abnormal_values, mode_val)
        df_rent['付款方式_cleaned'] = df_rent['付款方式_cleaned'].fillna(mode_val)
        
        # 有序编码
        payment_mapping = {
            '月付价': 1, '双月付价': 2, '季付价': 3, '半年付价': 6, '年付价': 12
        }
        X['付款方式_encoded'] = df_rent['付款方式_cleaned'].map(payment_mapping)
        
        # 特征工程优化
        # 1. 是否为长期付款（季付及以上）
        X['是长期付款'] = (X['付款方式_encoded'] >= 3).astype(bool)
        
        # 2. 付款方式独热编码
        payment_dummies = pd.get_dummies(df_rent['付款方式_cleaned'], prefix='付款方式').astype(bool)
        X = pd.concat([X, payment_dummies], axis=1)
        
        if verbose:
            print("加入付款方式特征：有序编码 + 长期付款标志 + 独热编码")
    
    return X

In [44]:

def rent_process_rental_method(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理租赁方式列 - 布尔编码"""
    if '租赁方式' in df_rent.columns:
        # 使用 map 替代 replace，避免向下转换警告
        rental_mapping = {'整租': True, '合租': False}
        X['是整租'] = df_rent['租赁方式'].fillna('整租').map(rental_mapping)
        
        # 确保数据类型为 bool
        X['是整租'] = X['是整租'].astype(bool)        
        
        if verbose:
            print("加入租赁方式特征：整租布尔标志")
    return X

In [45]:
def rent_process_elevator(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理电梯列 - 布尔编码"""
    if '电梯' in df_rent.columns:
        # 使用 map 替代 replace，避免向下转换警告
        elevator_mapping = {'有': True, '无': False}
        X['有电梯'] = df_rent['电梯'].fillna('无').map(elevator_mapping)
        
        # 确保数据类型为 bool
        X['有电梯'] = X['有电梯'].astype(bool)

        if verbose:
            print("加入电梯特征：有电梯布尔标志")
    return X

In [46]:
def rent_process_water_electricity(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理用水用电列 - 布尔编码"""
    # 用水
    if '用水' in df_rent.columns:
        # 使用 map 替代 replace，避免向下转换警告
        water_mapping = {'民水': True, '商水': False}
        X['是民水'] = df_rent['用水'].fillna('民水').map(water_mapping)
        X['是民水'] = X['是民水'].astype(bool)    
    # 用电
    if '用电' in df_rent.columns:
        # 使用 map 替代 replace，避免向下转换警告
        electricity_mapping = {'民电': True, '商电': False}
        X['是民电'] = df_rent['用电'].fillna('民电').map(electricity_mapping)
        X['是民电'] = X['是民电'].astype(bool)    
    if verbose and ('用水' in df_rent.columns or '用电' in df_rent.columns):
        print("加入水电特征：民水民电布尔标志")
    
    return X

In [47]:
def rent_process_lease_term(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理租期列 - 转换为月数 + 创建特征"""
    if '租期' in df_rent.columns:
        def term_to_months(text):
            nums = list(map(int, re.findall(r'\d+', str(text))))
            if not nums:
                return 6
            if '年' in str(text):
                if len(nums) == 1:
                    year = nums[0]
                    if '以上' in str(text):
                        return year * 12 + 12
                    elif '以内' in str(text):
                        return year * 12 / 2
                    else:
                        return year * 12
                else:
                    min_year, max_year = nums
                    return (min_year * 12 + max_year * 12) / 2
            elif '月' in str(text):
                if '~' in str(text):
                    min_month, max_month = nums
                    return (min_month + max_month) / 2
                else:
                    month = nums[0]
                    if '以上' in str(text):
                        return month + 3
                    else:
                        return month
            return 6
        
        X['租期月数'] = df_rent['租期'].fillna('1年以内').apply(term_to_months)
        
        # 特征工程优化
        # 1. 是否为长期租约（1年以上）
        X['是长期租约'] = (X['租期月数'] >= 12).astype(bool)
        
        # 2. 租期对数变换
        X['租期月数_log'] = safe_log_transform(X['租期月数'])
        
        if verbose:
            print("加入租期特征：月数 + 长期租约标志 + 对数变换")
    
    return X

In [48]:
def rent_process_building_age(df_rent: pd.DataFrame, X: pd.DataFrame, current_year: int = 2025, verbose: bool = True) -> pd.DataFrame:
    """处理建筑年代列 - 计算建筑年龄 + 分箱 + 交互特征"""
    if '建筑年代' in df_rent.columns:
        def parse_year(x):
            if pd.isna(x):
                return None
            nums = re.findall(r'\d{4}', str(x))
            if len(nums) == 1:
                return int(nums[0])
            elif len(nums) == 2:
                return (int(nums[0]) + int(nums[1])) / 2
            else:
                return None
        
        df_rent['建筑年代_parsed'] = df_rent['建筑年代'].apply(parse_year)
        median_year = df_rent['建筑年代_parsed'].median()
        df_rent['建筑年代_parsed'] = df_rent['建筑年代_parsed'].fillna(median_year)
        
        # 基础建筑年龄
        building_age = current_year - df_rent['建筑年代_parsed']
        X['建筑年龄'] = building_age
        
        # 特征工程优化
        # 1. 是否为新房（5年内）
        X['是新房'] = (building_age <= 5).astype(bool)
        
        # 2. 是否为老房（20年以上）
        X['是老房'] = (building_age > 20).astype(bool)
        
        if verbose:
            print("加入建筑年龄特征：数值 + 新旧标志")
    
    return X

In [49]:
def rent_process_building_counts(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理房屋总数和楼栋总数 - 数值化 + 创建密度特征"""
    # 房屋总数
    if '房屋总数' in df_rent.columns:
        X['房屋总数'] = df_rent['房屋总数'].str.replace('户', '', regex=False).astype(float)
        X['房屋总数'] = X['房屋总数'].fillna(X['房屋总数'].median())
    
    # 楼栋总数
    if '楼栋总数' in df_rent.columns:
        X['楼栋总数'] = df_rent['楼栋总数'].str.replace('栋', '', regex=False).astype(float)
        X['楼栋总数'] = X['楼栋总数'].fillna(X['楼栋总数'].median())
    
    # 特征工程优化 - 创建密度特征
    if '房屋总数' in X.columns and '楼栋总数' in X.columns:
        X['每栋户数'] = X['房屋总数'] / (X['楼栋总数'] + 0.01)
        X['每栋户数_log'] = safe_log_transform(X['每栋户数'])
    
    if verbose and ('房屋总数' in df_rent.columns or '楼栋总数' in df_rent.columns):
        print("加入楼栋房屋特征：数值 + 密度特征")
    
    return X

In [50]:
def rent_process_green_rate(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理绿化率列 - 数值化 + 分箱"""
    if '绿 化 率' in df_rent.columns:
        green_series = df_rent['绿 化 率'].str.replace('%', '', regex=False)
        green_numeric = pd.to_numeric(green_series, errors='coerce')
        
        # 处理异常值
        normal_green = green_numeric[(green_numeric <= 100) & (green_numeric.notna())]
        median_green = normal_green.median() if not normal_green.empty else 0
        green_filled = green_numeric.apply(lambda x: median_green if (pd.isna(x) or x > 100) else x)
        
        X['绿化率'] = green_filled
        
        # 特征工程优化
        # 1. 是否为高绿化率（>30%）
        X['是高绿化率'] = (green_filled > 30).astype(bool)
        
        if verbose:
            print("加入绿化率特征：数值 + 高绿化率标志")
    
    return X

In [51]:
def rent_process_floor_area_ratio(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理容积率列 - 数值化 + 分箱"""
    if '容 积 率' in df_rent.columns:
        far_series = pd.to_numeric(df_rent['容 积 率'], errors='coerce')
        median_far = far_series.median()
        far_filled = far_series.fillna(median_far)
        
        X['容积率'] = far_filled
        
        # 特征工程优化
        # 1. 是否为低容积率（<2.0）
        X['是低容积率'] = (far_filled < 2.0).astype(bool)
        
        if verbose:
            print("加入容积率特征：数值 + 低容积率标志")
    
    return X

In [52]:
def rent_process_property_fee(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理物业费列 - 解析数值 + 对数变换 + 分箱"""
    if '物 业 费' in df_rent.columns:
        def parse_fee(x):
            if pd.isna(x):
                return None
            num_str = str(x).replace('元/月/㎡', '').strip()
            if '-' in num_str:
                parts = num_str.split('-')
                if len(parts) == 2:
                    try:
                        return (float(parts[0]) + float(parts[1])) / 2
                    except:
                        return None
                return None
            else:
                try:
                    return float(num_str)
                except:
                    return None
        
        fee_parsed = df_rent['物 业 费'].apply(parse_fee)
        median_fee = fee_parsed.median() if fee_parsed.notna().any() else 0
        fee_filled = fee_parsed.fillna(median_fee)
        
        X['物业费'] = fee_filled
        
        # 特征工程优化
        # 1. 物业费对数变换
        X['物业费_log'] = safe_log_transform(fee_filled)
        
        if verbose:
            print("加入物业费特征：数值 + 对数")
    
    return X

In [53]:
def rent_process_water_supply(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理供水列 - 民/商布尔标志"""
    if '供水' in df_rent.columns:
        X['供水_民'] = df_rent['供水'].apply(lambda x: int('民' in str(x))).astype(bool)
        X['供水_商'] = df_rent['供水'].apply(lambda x: int('商' in str(x))).astype(bool)
        if verbose:
            print("加入供水特征：民/商布尔标志")
    return X

In [54]:
def rent_process_heating(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理供暖列 - 编码类型 + 独热编码"""
    if '供暖' in df_rent.columns:
        # 填充空值
        heating_series = df_rent['供暖'].fillna(0)
        
        # 编码函数
        def heating_to_num(text):
            if not isinstance(text, str):
                return 0
            if '集中供暖' in text:
                return 2
            elif '自采暖' in text:
                return 1
            else:
                return 0
        
        X['供暖_encoded'] = heating_series.apply(heating_to_num)
        
        # 供暖类型独热编码
        heating_dummies = pd.get_dummies(heating_series.astype(str), prefix='供暖').astype(bool)
        X = pd.concat([X, heating_dummies], axis=1)
        
        if verbose:
            print("加入供暖特征：有序编码 + 独热编码")
    
    return X

In [55]:
def rent_process_power_supply(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理供电列 - 民/商布尔标志"""
    if '供电' in df_rent.columns:
        X['供电_民'] = df_rent['供电'].apply(lambda x: int('民' in str(x))).astype(bool)
        X['供电_商'] = df_rent['供电'].apply(lambda x: int('商' in str(x))).astype(bool)
        if verbose:
            print("加入供电特征：民/商布尔标志")
    return X

In [56]:
def rent_process_gas_fee(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理燃气费列 - 解析数值 + 对数变换"""
    if '燃气费' in df_rent.columns:
        def parse_gas_fee(x):
            if pd.isna(x):
                return None
            num_str = str(x).replace('元/m³', '').replace('元/m3', '').strip()
            if '-' in num_str:
                parts = num_str.split('-')
                if len(parts) == 2:
                    try:
                        return (float(parts[0]) + float(parts[1])) / 2
                    except:
                        return None
                return None
            else:
                try:
                    return float(num_str)
                except:
                    return None
        
        gas_parsed = df_rent['燃气费'].apply(parse_gas_fee)
        median_gas = gas_parsed.median() if gas_parsed.notna().any() else 0
        gas_filled = gas_parsed.fillna(median_gas)
        
        X['燃气费'] = gas_filled
        
        # 特征工程优化
        X['燃气费_log'] = safe_log_transform(gas_filled)
        
        if verbose:
            print("加入燃气费特征：数值 + 对数变换")
    
    return X

In [57]:
def rent_process_parking(df_rent: pd.DataFrame, X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """处理停车位列 - 数值化 + 对数变换"""
    if '停车位' in df_rent.columns:
        parking_series = pd.to_numeric(df_rent['停车位'], errors='coerce')
        median_parking = parking_series.median() if parking_series.notna().any() else 0
        parking_filled = parking_series.fillna(median_parking)
        
        X['停车位'] = parking_filled
        
        # 特征工程优化
        X['停车位_log'] = safe_log_transform(parking_filled)
        
        if verbose:
            print("加入停车位特征：数值 + 对数变换 + 有无标志")
    
    return X

In [58]:
def create_interaction_features(X: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """创建特征交互项"""
    
    # 面积与房间数的交互
    if '面积' in X.columns and '室' in X.columns:
        X['每室面积'] = X['面积'] / (X['室'] + 0.01)
        X['每室面积_log'] = safe_log_transform(X['每室面积'])
    
    # 建筑年龄与装修等级的交互
    if '建筑年龄' in X.columns and '装修等级' in X.columns:
        X['老房新装'] = ((X['建筑年龄'] > 20) & (X['装修等级'] >= 4)).astype(bool)
    
    # 面积与楼层的交互
    if '面积' in X.columns and '楼层位置比' in X.columns:
        X['面积楼层交互'] = X['面积'] * X['楼层位置比']
    
    # 房间数与装修等级的交互
    if '室' in X.columns and '装修等级' in X.columns:
        X['大户型精装'] = ((X['室'] >= 3) & (X['装修等级'] >= 4)).astype(bool)
    
    # 面积与朝向得分的交互
    if '面积' in X.columns and '朝向得分' in X.columns:
        X['面积朝向交互'] = X['面积'] * X['朝向得分']
    
    if verbose:
        print("创建交互特征完成")
    
    return X

In [59]:
# 步骤 1: 数据加载 (Rent) 
print("--- 步骤 1: 数据加载 (仅rent) ---")
try:
    # 您已经加载了训练集，现在加载测试集
    df_test_rent_raw = pd.read_csv("data/ruc_Class25Q2_test_rent.csv")
    print(f"Rent 训练数据已加载: {df_rent.shape}")
    print(f"Rent 测试数据加载成功: {df_test_rent_raw.shape}")
except FileNotFoundError:
    print("错误：未找到 Rent 测试数据文件。请确保文件在 'data/' 目录下。")
    exit()

# --- 步骤 2: 存储原始信息与分离 ---
print("\n--- 步骤 2: 存储原始信息与分离 ---")
n_train_rent = df_rent.shape[0]  # 您已经加载的训练集行数
print(f"Rent 训练集原始行数: {n_train_rent}")

# 分离目标变量 (rent) - 从您已经加载的df_rent中
if 'Price' in df_rent.columns:
    y_train_rent = df_rent['Price'].copy()
    # 对目标变量进行对数变换 
    y_train_ln_rent = np.log1p(y_train_rent)
    print(f"已分离 Rent 目标变量 (y_train_rent, y_train_ln_rent)，长度: {len(y_train_rent)}")
else:
    print("警告: 'Price' 列在训练集中未找到。")
    y_train_rent = None
    y_train_ln_rent = None

# 分离测试集 ID
if 'ID' in df_test_rent_raw.columns:
    test_ids_rent = df_test_rent_raw['ID'].copy()
    print(f"已分离 Rent 测试集 ID (test_ids_rent)，长度: {len(test_ids_rent)}")
else:
    print("警告: 'ID' 列在测试集中未找到。")
    test_ids_rent = None

# --- 步骤 3: 合并数据集以便统一处理 ---
print("\n--- 步骤 3: 合并 Rent 训练集与测试集 ---")
# 从训练集中移除 Price 列，从测试集中移除 ID 列
df_train_to_concat = df_rent.drop(columns=['Price'], errors='ignore')
df_test_to_concat = df_test_rent_raw.drop(columns=['ID'], errors='ignore')

# 添加来源标识
df_train_to_concat['source'] = 'train'
df_test_to_concat['source'] = 'test'

# 合并
df_rent_combined = pd.concat([df_train_to_concat, df_test_to_concat], ignore_index=True)
print(f"Rent 数据集合并完成。合并后 df_rent_combined 形状: {df_rent_combined.shape}")

# 清理原始数据框以释放内存
del df_test_rent_raw, df_train_to_concat, df_test_to_concat
import gc
gc.collect()

--- 步骤 1: 数据加载 (仅rent) ---
Rent 训练数据已加载: (93365, 46)
Rent 测试数据加载成功: (9773, 46)

--- 步骤 2: 存储原始信息与分离 ---
Rent 训练集原始行数: 93365
已分离 Rent 目标变量 (y_train_rent, y_train_ln_rent)，长度: 93365
已分离 Rent 测试集 ID (test_ids_rent)，长度: 9773

--- 步骤 3: 合并 Rent 训练集与测试集 ---
Rent 数据集合并完成。合并后 df_rent_combined 形状: (103138, 46)


98

In [60]:

# ==================== 主函数 ====================

def preprocess_rent(
    df_price: pd.DataFrame,
    X: Optional[pd.DataFrame] = None,
    y_target: Optional[pd.Series] = None,  # 新增：目标变量
    current_year: int = 2025,
    verbose: bool = True,
    create_interactions: bool = True,
    n_train: Optional[int] = None,
    create_geo_features: bool = True,
    apply_target_encoding: bool = True,  # 新增：是否应用目标编码
    location_cols_to_encode: List[str] = None,  # 新增：目标编码的列
) -> pd.DataFrame:
    """
    对租房数据做统一的预处理 & 特征工程，包含丰富的特征变换。
    
    参数:
      - df_rent: 原始 pandas.DataFrame
      - X: 初始特征表
      - current_year: 用于计算建筑年龄的当前年份
      - verbose: 是否打印中间信息
      - create_interactions: 是否创建交互特征
      - create_polynomials: 是否创建多项式特征
    """
    # 复制原始数据，避免修改输入
    df_rent_processed = df_rent.copy()
    
    # 初始化特征表X
    if X is None:
        X = pd.DataFrame(index=df_rent_processed.index)
    
    # --------------------------
    # 1. 删除无关列
    # --------------------------
    drop_cols = [
        '环线位置',  'coord_x', 'coord_y', '年份',
        '客户反馈',
        '物业类别', '产权描述', 
        '配套设施', '物业公司', '物业办公电话', '开发商', '车位', '停车费用',
        '采暖', '供热费'
    ]
    
    cols_to_drop = [col for col in drop_cols if col in df_rent_processed.columns]
    df_rent_processed = df_rent_processed.drop(columns=cols_to_drop, errors="ignore")
    
    if verbose:
        print("删除后数据维度:", df_rent_processed.shape)
        print("\n缺失值统计（前20）：")
        print(df_rent_processed.isnull().sum().sort_values(ascending=False).head(20))



     # 处理区县列
    if '区县' in df_rent_processed.columns:
        df_rent_processed = handle_district(df_rent_processed)
        X['区县'] = df_rent_processed['区县']
    
    # 处理板块列
    if '板块' in df_rent_processed.columns:
        df_rent_processed = handle_plate(df_rent_processed)
        X['板块'] = df_rent_processed['板块']

    # --------------------------
    # 2. 调用各列处理函数
    # --------------------------
    X = rent_process_city(df_rent_processed, X, verbose)
    X = rent_process_room_layout(df_rent_processed, X, verbose)
    X = rent_process_decoration(df_rent_processed, X, verbose)
    X = rent_process_floor(df_rent_processed, X, verbose)
    X = rent_process_area(df_rent_processed, X, verbose)
    X = rent_process_direction(df_rent_processed, X, verbose)
    X = rent_process_structure(df_rent_processed, X, verbose)
    X = rent_process_transaction_time(df_rent_processed, X, current_year, verbose)
    X = rent_process_payment_method(df_rent_processed, X, verbose)
    X = rent_process_rental_method(df_rent_processed, X, verbose)
    X = rent_process_elevator(df_rent_processed, X, verbose)
    X = rent_process_water_electricity(df_rent_processed, X, verbose)
    X = rent_process_lease_term(df_rent_processed, X, verbose)
    X = rent_process_building_age(df_rent_processed, X, current_year, verbose)
    X = rent_process_building_counts(df_rent_processed, X, verbose)
    X = rent_process_green_rate(df_rent_processed, X, verbose)
    X = rent_process_floor_area_ratio(df_rent_processed, X, verbose)
    X = rent_process_property_fee(df_rent_processed, X, verbose)
    X = rent_process_water_supply(df_rent_processed, X, verbose)
    X = rent_process_heating(df_rent_processed, X, verbose)
    X = rent_process_power_supply(df_rent_processed, X, verbose)
    X = rent_process_gas_fee(df_rent_processed, X, verbose)
    X = rent_process_parking(df_rent_processed, X, verbose)




    # --------------------------
    # 4. 创建地理空间特征（新增）
    # --------------------------
    if create_geo_features and n_train is not None:
        # 将df_rent_processed中的地理位置列复制到X中，以便地理处理函数使用
        geo_cols = ['城市', 'lon', 'lat']
        for col in geo_cols:
            if col in df_rent_processed.columns and col not in X.columns:
                X[col] = df_rent_processed[col]
        
        # 应用地理处理函数
        X = compute_city_center_and_distances(X, n_train, verbose)
        X = create_geo_clusters(X, n_train=n_train, n_clusters=5, verbose=verbose)
        
        # 清理中间的地理位置列
        for col in geo_cols:
            if col in X.columns:
                X = X.drop(columns=[col])

    # --------------------------
    # 5. 应用目标编码（新增）
    # --------------------------
    if apply_target_encoding and y_target is not None and n_train is not None:
        if location_cols_to_encode is None:
            location_cols_to_encode = ['城市', '区县', '板块']
        
        # 确保目标编码的列存在于X中
        existing_location_cols = [col for col in location_cols_to_encode if col in X.columns]
        if existing_location_cols:
            if verbose:
                print(f"\n应用目标编码到列: {existing_location_cols}")
            X = apply_target_encoding_combined(X, y_target, n_train, existing_location_cols)
        elif verbose:
            print(f"\n警告: 目标编码列 {location_cols_to_encode} 在特征矩阵中不存在")



    # --------------------------
    # 6. 创建高级特征
    # --------------------------
    if create_interactions:
        X = create_interaction_features(X, verbose)
    
    #if create_polynomials:
    #    X = create_polynomial_features(X, verbose)

    # --------------------------
    # 4. 处理缺失值
    # --------------------------
    # 数值列用中位数填充
    numeric_cols = X.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if X[col].isna().any():
            median_val = X[col].median()
            X[col] = X[col].fillna(median_val)
            if verbose:
                print(f"填充数值列 '{col}' 的缺失值: {median_val}")

    # 布尔列用众数填充
    bool_cols = X.select_dtypes(include=[bool]).columns
    for col in bool_cols:
        if X[col].isna().any():
            mode_val = X[col].mode()[0] if not X[col].mode().empty else False
            X[col] = X[col].fillna(mode_val)
            if verbose:
                print(f"填充布尔列 '{col}' 的缺失值: {mode_val}")

    # 对象型列用众数填充（新增）
    object_cols = X.select_dtypes(include=['object']).columns
    for col in object_cols:
        if X[col].isna().any():
            mode_val = X[col].mode()[0] if not X[col].mode().empty else '未知'
            X[col] = X[col].fillna(mode_val)
            if verbose:
                print(f"填充对象列 '{col}' 的缺失值: {mode_val}")
    # --------------------------
    # 5. 最终信息展示
    # --------------------------
    if verbose:
        print("\n=== 特征工程完成 ===")
        print(f"最终 X 形状：{X.shape}")
        print(f"数值型特征: {len(X.select_dtypes(include=[np.number]).columns)}")
        print(f"布尔型特征: {len(X.select_dtypes(include=[bool]).columns)}")
        
        # 显示前10个数值特征的统计信息
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            print(f"\n前10个数值特征的统计信息:")
            print(X[numeric_cols[:10]].describe())

    return X

In [61]:
# ============================================================================
# 完整的训练和预测流程（整合版本）- 为四个模型分别生成预测结果
# ============================================================================

def complete_training_pipeline(
    df_combined: pd.DataFrame,
    y_train_target: pd.Series,
    n_train: int,
    test_ids: pd.Series,
    current_year: int = 2025,
    apply_outlier_detection: bool = False,
    outlier_threshold: float = 0.99,
    test_size: float = 0.2,
    random_state: int = 42,
    verbose: bool = True
):
    """
    完整的房价预测流程：特征工程 → 异常值处理 → 模型训练 → 预测
    为四个模型（OLS、LASSO、Ridge、ElasticNet）分别生成预测结果
    """
    
    print("=" * 80)
    print("开始完整房价预测流程 - 四个模型分别预测")
    print("=" * 80)
    
    # 步骤1: 特征工程
    print("\n--- 步骤1: 特征工程 ---")
    X_processed = preprocess_price(
        df_price=df_combined,
        y_target=y_train_target,
        n_train=n_train,
        current_year=current_year,
        create_geo_features=True,
        apply_target_encoding=True,
        location_cols_to_encode=['城市', '区县', '板块'],
        verbose=verbose
    )
    
    print(f"特征工程完成，特征矩阵形状: {X_processed.shape}")
    
    # 步骤2: 分离训练集和测试集
    print("\n--- 步骤2: 分离训练集和测试集 ---")
    X_train_full = X_processed.iloc[:n_train].copy()
    X_test_final = X_processed.iloc[n_train:].copy()
    
    y_train_full = y_train_target.copy()
    y_train_log = np.log(y_train_full)
    
    print(f"训练集: {X_train_full.shape}")
    print(f"测试集: {X_test_final.shape}")
    
    # 步骤3: 异常值处理
    if apply_outlier_detection:
        print("\n--- 步骤3: 异常值检测与处理 ---")
        X_clean, y_clean, outlier_info = detect_and_remove_outliers(
            X_train_full, y_train_full, 
            method='quantile', 
            threshold=outlier_threshold,
            verbose=verbose
        )
        y_clean_log = np.log(y_clean)
    else:
        print("\n--- 步骤3: 跳过异常值检测 ---")
        X_clean, y_clean = X_train_full.copy(), y_train_full.copy()
        y_clean_log = y_train_log.copy()
        outlier_info = {}
    
    print(f"清理后训练集: {X_clean.shape}")
    
    # 步骤4: 数据准备和标准化
    print("\n--- 步骤4: 数据准备和标准化 ---")
    
    # 4.1 划分训练测试集
    X_train, X_val, y_train, y_val, y_train_log, y_val_log = prepare_model_data(
        X_clean, y_clean, test_size=test_size, random_state=random_state
    )
    
    print(f"训练集: {X_train.shape}, 验证集: {X_val.shape}")
    
    # 4.2 标准化特征
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns
    
    scaler = StandardScaler()
    X_train_scaled = X_train.copy()
    X_val_scaled = X_val.copy()
    X_test_final_scaled = X_test_final.copy()
    
    X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])
    X_test_final_scaled[numeric_cols] = scaler.transform(X_test_final[numeric_cols])
    
    print(f"标准化完成:")
    print(f"  训练集: {X_train_scaled.shape}")
    print(f"  验证集: {X_val_scaled.shape}")
    print(f"  测试集: {X_test_final_scaled.shape}")
    
    # 步骤5: 模型训练和评估
    print("\n--- 步骤5: 模型训练和评估 ---")
    
    all_results, best_model_info, models_dict = train_and_evaluate_models(
        X_train_scaled, X_val_scaled, y_train, y_val, y_train_log, y_val_log, verbose
    )
    
    # 步骤6: 在完整训练集上重新训练所有模型
    print("\n--- 步骤6: 在完整训练集上重新训练所有模型 ---")
    
    # 在完整清理后的数据上重新标准化
    X_full_clean_scaled = X_clean.copy()
    X_full_clean_scaled[numeric_cols] = scaler.transform(X_clean[numeric_cols])
    
    print(f"完整训练集最终形状: {X_full_clean_scaled.shape}")
    
    # 重新训练所有模型
    final_models = {}
    for model_name, model in models_dict.items():
        print(f"重新训练 {model_name} 模型...")
        # 克隆模型以避免修改原始模型
        from sklearn.base import clone
        final_model = clone(model)
        final_model.fit(X_full_clean_scaled, y_clean_log)
        final_models[model_name] = final_model
    
    print("所有模型重新训练完成!")
    
    # 步骤7: 特征重要性分析（使用最佳模型）
    print("\n--- 步骤7: 特征重要性分析 ---")
    
    best_model_name = best_model_info['model_name']
    best_final_model = final_models[best_model_name]
    
    if hasattr(best_final_model, 'coef_'):
        feature_importance = pd.DataFrame({
            'feature': X_full_clean_scaled.columns,
            'coefficient': best_final_model.coef_,
            'abs_coefficient': np.abs(best_final_model.coef_)
        }).sort_values('abs_coefficient', ascending=False)
        
        print(f"\n最佳模型 ({best_model_name}) 前20个最重要特征:")
        print("-" * 60)
        for idx, row in feature_importance.head(20).iterrows():
            sign = "↓" if row['coefficient'] < 0 else "↑"
            print(f"  {row['feature']:<40} {row['coefficient']:>8.4f} {sign}")
    
    # 步骤8: 为所有模型生成预测结果
    print("\n--- 步骤8: 为所有模型生成预测结果 ---")
    
    timestamp = pd.Timestamp.now().strftime("%m%d%H%M")
    output_files = {}
    
    for model_name, model in final_models.items():
        print(f"\n使用 {model_name} 模型进行预测...")
        
        # 进行预测
        y_test_pred_log = model.predict(X_test_final_scaled)
        y_test_pred = np.exp(y_test_pred_log)
        
        # 生成输出文件
        output = pd.DataFrame({
            "ID": test_ids,
            "Price": y_test_pred
        })
        
        # 保存预测结果
        output_file = f"data/final_predicted_rent_{model_name}_{timestamp}.csv"
        output.to_csv(output_file, index=False, encoding="utf-8-sig")
        output_files[model_name] = output_file
        
        print(f"✅ {model_name} 预测完成！结果已保存至 {output_file}")
        print(f"{model_name} 预测价格统计:")
        print(f"  最小值: {y_test_pred.min():,.2f}")
        print(f"  最大值: {y_test_pred.max():,.2f}")
        print(f"  平均值: {y_test_pred.mean():,.2f}")
        print(f"  中位数: {np.median(y_test_pred):,.2f}")
        
        # 只显示最佳模型的预测结果预览
        if model_name == best_model_name:
            print(f"\n最佳模型 {model_name} 预测结果预览:")
            print(output.head(10))
    
    # 步骤9: 模型性能比较
    print("\n--- 步骤9: 模型性能比较 ---")
    
    print(f"\n所有模型在验证集上的性能比较:")
    print("-" * 80)
    print(f"{'模型':<12} {'测试集MAE':<12} {'测试集RMAE':<12} {'测试集R²':<10}")
    print("-" * 80)
    
    for model_name, results in all_results.items():
        print(f"{model_name:<12} {results['test_mae']:>10,.2f} {results['test_rmae']:>10.2f}% {results['test_r2']:>9.4f}")
    
    print(f"\n🎯 最佳模型: {best_model_name}")
    print(f"最佳测试集MAE: {best_model_info['test_mae']:,.2f}")
    print(f"最佳测试集RMAE: {best_model_info['test_rmae']:.2f}%")
    
    # 返回所有重要信息
    return {
        'final_models': final_models,
        'scaler': scaler,
        'best_model_info': best_model_info,
        'all_results': all_results,
        'models_dict': models_dict,
        'output_files': output_files,
        'feature_importance': feature_importance if 'feature_importance' in locals() else None,
        'outlier_info': outlier_info,
        'X_processed_shape': X_processed.shape,
    }

In [62]:
result_modified = complete_training_pipeline(
    df_combined=df_rent_combined,  # 您合并的数据
    y_train_target=y_train_rent,    # 训练集目标变量
    n_train=n_train_rent,           # 训练集大小
    test_ids=test_ids_rent,         # 测试集ID
    current_year=2025,
    apply_outlier_detection=True,
    outlier_threshold=0.99,
    test_size=0.2,
    random_state=42,
    verbose=True
)

print(f"\n🎉 完整流程执行完毕！")
print(f"最佳模型: {result_modified['best_model_info']['model_name']}")
print(f"测试集MAE: {result_modified['best_model_info']['test_mae']:,.2f}")
print(f"测试集RMAE: {result_modified['best_model_info']['test_rmae']:.2f}%")

print(f"\n所有模型的输出文件:")
for model_name, file_path in result_modified['output_files'].items():
    print(f"  {model_name}: {file_path}")

开始完整房价预测流程 - 四个模型分别预测

--- 步骤1: 特征工程 ---
删除列： ['环线位置', '年份', '客户反馈', '产权描述', 'coord_x', 'coord_y', '物业公司', '供热费', '物业办公电话', '开发商', '停车费用']
删除后数据维度: (103138, 35)

缺失值统计（前20）：
车位       78308
装修       72427
采暖       69191
供暖       66009
租期       49977
配套设施     32835
建筑年代     29190
停车位      28256
燃气费      27856
绿 化 率    27124
容 积 率    26797
物 业 费    24725
供电       23749
供水       23744
建筑结构     23232
物业类别     22546
付款方式     20531
用水       19592
用电       19141
板块        6028
dtype: int64
处理 [区县]...
  已将 '区县' 填充缺失值并转换为 'string' 类型。
  处理后 '区县' 的唯一值 (示例): ['81.0' '7.0' '68.0' '123.0' '95.0' '109.0' '112.0' '62.0' '未知' '5.0']
处理 [板块]...
  已将 '板块' 填充缺失值并转换为 'string' 类型。


加入城市独热编码，新增列数： 12
加入建筑结构独热编码
加入建筑年龄特征：数值 + 分箱 + 新房标志
加入楼栋房屋特征：数值 + 密度特征
加入绿化率特征：数值
加入容积率特征：数值 + 分箱
加入物业费特征：数值 + 对数
加入供水_民/供水_商
加入供暖特征：有序编码 + 独热编码
加入供电_民/供电_商
加入燃气费（数值化）
加入停车位（数值化、缺失用中位数填充）
开始计算地理空间特征 (距离中心)...
  计算了 12 个城市的中心点 (基于训练集)。


C:\Users\lenovo\AppData\Local\Temp\ipykernel_19996\3536126342.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_out['距离中心_公里'].fillna(median_dist_train, inplace=True)
d:\APP\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
d:\APP\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning

  计算了 '距离中心_公里'，并用训练集中位数 (13.78 km) 填充了 NaN。
地理距离特征创建完毕。
开始创建地理聚类 (每个城市 5 个簇)...


d:\APP\Anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


地理聚类特征创建完毕。

应用目标编码到列: ['区县', '板块']
处理 ['区县', '板块'] (K-Fold Target Encoding, n_splits=6)...
  将对以下存在的列进行编码: ['区县', '板块']
  计算了基于 93365 训练样本的完整均值图谱。
  已将完整图谱应用于 9773 个测试样本。
    填充训练集中 K-Fold 后剩余的 51 个 NaN...
  K-Fold Target Encoding 完成。新特征: 'Location_Target_Encoded'。
创建交互特征完成

=== 特征工程完成 ===
最终 X 列数：115
数值型特征: 15
布尔型特征: 100
对象型特征: 0
特征工程完成，特征矩阵形状: (103138, 115)

--- 步骤2: 分离训练集和测试集 ---
训练集: (93365, 115)
测试集: (9773, 115)

--- 步骤3: 异常值检测与处理 ---
异常值检测与处理
检测数值型列 (15个): ['建筑年龄', '房屋总数', '楼栋总数', '每栋户数', '每栋户数_log', '绿化率', '容积率', '物业费', '物业费_log', '供暖_encoded', '燃气费', '停车位', '距离中心_公里', '距离中心_公里_平方', 'Location_Target_Encoded']
跳过布尔型列 (100个)

检测到异常值行数: 14719
占总数据比例: 15.77%
清理后数据形状: (78646, 115)
清理后目标变量形状: (78646,)
清理后训练集: (78646, 115)

--- 步骤4: 数据准备和标准化 ---
数据准备阶段
--------------------------------------------------
目标变量统计:
  原始范围: [17938.07, 1449604.24]
  原始均值: 467649.34, 标准差: 297318.21
  对数化后: [9.79, 14.19]

数据划分:
  训练集: 62916 样本
  验证集: 15730 样本
  特征数: 115
训练集: (62916, 115), 验证集: (15730, 115)
标准化